# SkyGuard AI — GPU Iteration 7

## Climate-calibrated weather transfer and causal TCN consensus

Iteration 6 correctly refused three unsafe promotions. It established that unknown-cadence archive gaps must remain advisory, found a weather challenger that missed the station-macro gate by a small margin, and showed that a point-level weak-fault rescue did not transfer to October or pseudo-unseen stations.

Iteration 7 therefore changes the representation rather than relaxing any gate:

1. Freeze the heartbeat-contract communication policy.
2. Convert station-invariant weather scores to climate-cluster percentiles and train sparse L1/L2 meta-calibrators.
3. Train a small causal temporal convolutional network on relative, causal sequences only.
4. Require tree–TCN consensus and incident persistence for weak-fault rescue.
5. Promote nothing unless May–September discovery, October confirmation, and pseudo-unseen station confirmation all pass.

The 2024 and 2025 benchmark files remain sealed. All earlier sections are reconstruction checkpoints and must be run in order.


## Run instructions

1. Keep `SkyGuard_GPU_Data_Bundle.zip` in `/content/drive/MyDrive/SkyGuard_AI_GPU/`.
2. In Colab select **Runtime → Change runtime type → T4 GPU**.
3. Run the notebook from the first cell through the final Iteration 7 cell without skipping historical reconstruction cells.
4. Keep `UNLOCK_FINAL_TESTS=False` and `REUSE_SAVED_MODELS=True`.
5. Expected runtime is approximately 45–100 minutes when prior CatBoost/LightGBM models already exist; rebuilding all historical models can take longer.
6. Return only the twelve Iteration 7 files printed by the final cell. Do not send or use any 2025 label file for tuning.


In [ ]:
!pip -q install catboost==1.2.10 lightgbm==4.6.0 scikit-learn==1.7.2 pyarrow==21.0.0 psutil==7.0.0
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive/SkyGuard_AI_GPU')
BUNDLE_ZIP=DRIVE_ROOT/'SkyGuard_GPU_Data_Bundle.zip'
DATA_ROOT=DRIVE_ROOT/'SkyGuard_GPU_Data_Bundle'
ITER1=DRIVE_ROOT/'experiments'/'iteration_01_detection'
ARTIFACT_ROOT=DRIVE_ROOT/'experiments'/'iteration_02_weak_fault_rescue'
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)

UNLOCK_FINAL_TESTS=False
REUSE_SAVED_MODELS=True
SEEDS=[17,29,41,53,67]
SPECIALIST_SEEDS=[17,41,67]
print('Iteration 1:',ITER1)
print('Iteration 2:',ARTIFACT_ROOT)


In [ ]:
import os,json,time,math,hashlib,zipfile,warnings,platform
import joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt,psutil
from IPython.display import display
from sklearn.metrics import average_precision_score,precision_score,recall_score,f1_score,confusion_matrix
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import torch

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',50)
pd.set_option('display.float_format',lambda x:f'{x:,.5f}')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print({'python':platform.python_version(),'device':DEVICE,
       'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,
       'ram_gb':round(psutil.virtual_memory().total/2**30,2)})
assert DEVICE=='cuda','Select a GPU runtime.'

if not DATA_ROOT.exists():
    assert BUNDLE_ZIP.exists(),f'Missing {BUNDLE_ZIP}'
    with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(DRIVE_ROOT)


## 1. Load exact development partitions and Phase 10 contract

In [ ]:
FEATURE_DIR=DATA_ROOT/'data'/'features_phase10'
BASELINE_FILE=DATA_ROOT/'models'/'phase10_final.joblib'

def load_table(name):
    frame=pd.read_csv(FEATURE_DIR/f'{name}_features.csv.gz',low_memory=False)
    frame['station_id']=frame.station_id.astype(str)
    frame['emitted_timestamp_utc']=pd.to_datetime(frame.emitted_timestamp_utc,utc=True)
    frame['episode_id']=frame.episode_id.fillna('').astype(str)
    return frame.sort_values(['station_id','emitted_timestamp_utc','row_id']).reset_index(drop=True)

train=load_table('train'); validation=load_table('validation')
train['dev_split']='train'
ts=validation.emitted_timestamp_utc
validation['dev_split']=np.select(
    [ts<'2023-05-01',ts<'2023-07-01',ts<'2023-10-01'],
    ['tune_model','block_may_jun','block_jul_sep'],default='block_oct_dec')
episode_part=(validation.loc[validation.episode_id.ne('')].sort_values('emitted_timestamp_utc')
              .groupby('episode_id').dev_split.first())
mask=validation.episode_id.ne('')
validation.loc[mask,'dev_split']=validation.loc[mask,'episode_id'].map(episode_part)
dev=pd.concat([train,validation],ignore_index=True)
del train,validation
dev=dev.loc[dev.available_to_detector.eq(1)].copy().reset_index(drop=True)

bundle=joblib.load(BASELINE_FILE); FEATURES=list(bundle['event_features'])
assert len(FEATURES)==108 and not ({'temperature_dewpoint_spread_c','hour_sin','hour_cos','day_of_year_sin','day_of_year_cos'}&set(FEATURES))
assert dev.loc[dev.episode_id.ne('')].groupby('episode_id').dev_split.nunique().max()==1
display(dev.groupby('dev_split').agg(rows=('row_id','size'),fault_rows=('is_anomaly','sum'),
                                     episodes=('episode_id',lambda s:s[s.ne('')].nunique()),stations=('station_id','nunique')))


## 2. Exact Phase 10 and Iteration 1 CatBoost scores

In [ ]:
event_model=bundle['event_model']; classes=list(event_model.classes_)
fault_index=classes.index('sensor_fault'); weather_index=classes.index('genuine_weather')
proba=event_model.predict_proba(dev[FEATURES].replace([np.inf,-np.inf],np.nan))
dev['phase10_fault']=proba[:,fault_index]; dev['phase10_weather']=proba[:,weather_index]
PHASE10_THRESHOLD=float(bundle['policy']['known_station']['threshold'])

def load_cat_models(prefix,seeds):
    models=[]
    for seed in seeds:
        path=ITER1/f'{prefix}_seed{seed}.cbm'
        assert path.exists(),f'Missing {path}. Run Iteration 1 CatBoost cell or restore Drive artifacts.'
        model=CatBoostClassifier(); model.load_model(path); models.append(model)
    return models

fault_models=load_cat_models('cat_fault',SEEDS)
weather_models=load_cat_models('cat_weather',SEEDS)
X=dev[FEATURES]
fault_seed_scores=np.column_stack([m.predict_proba(X)[:,1] for m in fault_models])
weather_seed_scores=np.column_stack([m.predict_proba(X)[:,1] for m in weather_models])
dev['cat_fault_mean']=fault_seed_scores.mean(1)
dev['cat_fault_median']=np.median(fault_seed_scores,axis=1)
dev['cat_weather_mean']=weather_seed_scores.mean(1)
dev['cat_weather_median']=np.median(weather_seed_scores,axis=1)
print('Loaded',len(fault_models),'fault and',len(weather_models),'weather models.')


## 3. Strict metrics and fast causal alert policies

In [ ]:
def point_metrics(y,p,score):
    y=np.asarray(y,int); p=np.asarray(p,bool); score=np.asarray(score,float)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),
            'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),
            'f1':f1_score(y,p,zero_division=0),'auprc':average_precision_score(y,score)}

def predicted_events(group,pred_col):
    g=group.sort_values('emitted_timestamp_utc'); times=g.emitted_timestamp_utc.tolist(); pred=g[pred_col].to_numpy(bool)
    dt=g.emitted_timestamp_utc.diff().dt.total_seconds().div(60); positive=dt[dt.gt(0)]
    gap=max(60,2.5*(float(positive.median()) if len(positive) else 60))
    events=[]; start=None; previous=None
    for pos in np.flatnonzero(pred):
        separated=(previous is None or pos!=previous+1 or (times[pos]-times[previous]).total_seconds()/60>gap)
        if separated:
            if start is not None: events.append((times[start],times[previous]))
            start=pos
        previous=pos
    if start is not None: events.append((times[start],times[previous]))
    return events

def event_metrics(frame,pred_col):
    truth=[]
    labelled=frame.loc[frame.is_anomaly.eq(1)&frame.episode_id.ne('')]
    for (station,episode),g in labelled.groupby(['station_id','episode_id']):
        truth.append((station,episode,g.emitted_timestamp_utc.min(),g.emitted_timestamp_utc.max(),g.anomaly_type.mode().iloc[0]))
    predictions=[]; station_days=0
    for station,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); station_days+=max((g.emitted_timestamp_utc.iloc[-1]-g.emitted_timestamp_utc.iloc[0]).total_seconds()/86400,1/24)
        predictions.extend((station,a,b) for a,b in predicted_events(g,pred_col))
    used=set(); hits=[]; delays=[]; per_fault={}
    for station,episode,start,end,fault in truth:
        candidates=[(i,p) for i,p in enumerate(predictions) if i not in used and p[0]==station and p[1]<=end and p[2]>=start]
        hit=bool(candidates)
        if hit:
            i,p=min(candidates,key=lambda item:item[1][1]); used.add(i); delays.append(max(0,(max(start,p[1])-start).total_seconds()/60))
        hits.append(hit); per_fault.setdefault(fault,[]).append(hit)
    tp=sum(hits); fp=len(predictions)-len(used); fn=len(truth)-tp
    ep=tp/max(tp+fp,1); er=tp/max(tp+fn,1)
    return {'true_episodes':len(truth),'predicted_episodes':len(predictions),'event_precision':ep,'event_recall':er,
            'event_f1':2*ep*er/max(ep+er,1e-12),'false_alarm_episodes_per_station_day':fp/max(station_days,1e-12),
            'delay_median_min':float(np.median(delays)) if delays else None,'delay_p90_min':float(np.quantile(delays,.9)) if delays else None,
            'delay_mean_min':float(np.mean(delays)) if delays else None,
            'per_fault_episode_recall':{k:float(np.mean(v)) for k,v in sorted(per_fault.items())}}

def hysteresis(frame,score_col,start,cont):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); active=False; out=np.zeros(len(g),bool)
        for pos,score in enumerate(g[score_col].fillna(0).to_numpy(float)):
            if not active and score>=start: active=True
            elif active and score<cont: active=False
            out[pos]=active
        result.loc[g.index]=out
    return result

def persistent_signal(frame,score_col,threshold,min_points):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); run=0; out=np.zeros(len(g),bool)
        for pos,score in enumerate(g[score_col].fillna(0).to_numpy(float)):
            run=run+1 if score>=threshold else 0
            out[pos]=run>=min_points
        result.loc[g.index]=out
    return result

def evaluate(frame,score_col,pred_col): return {**point_metrics(frame.is_anomaly,frame[pred_col],frame[score_col]),**event_metrics(frame,pred_col)}


## 4. Train focused frozen, bias, and drift specialists

Each specialist uses a small causal feature family and three fixed seeds. Models train on 2022, use January–April 2023 only for early stopping, and never see the three robust policy blocks during fitting.


In [ ]:
FROZEN_FEATURES=[c for c in FEATURES if any(token in c for token in [
    'frozen_run_length','rolling_mad_24h','rolling_median_24h','delta1','rate_per_hour','neighbor_'])]
BIAS_FEATURES=[c for c in FEATURES if any(token in c for token in [
    'cusum_','climatology_residual','ewma_residual','neighbor_residual','regional_agreement'])]
DRIFT_FEATURES=[c for c in FEATURES if any(token in c for token in [
    '_slope_','cusum_','monotonic_run','climatology_residual','neighbor_residual','regional_'])]
SPECIALIST_FEATURES={'frozen_sensor':FROZEN_FEATURES,'bias':BIAS_FEATURES,'drift':DRIFT_FEATURES}
print({k:len(v) for k,v in SPECIALIST_FEATURES.items()})
assert all(len(v)>=12 for v in SPECIALIST_FEATURES.values())

train_mask=dev.dev_split.eq('train'); tune_mask=dev.dev_split.eq('tune_model')
specialist_models={}; specialist_history={}
for fault,features in SPECIALIST_FEATURES.items():
    y_train=dev.loc[train_mask,'anomaly_type'].eq(fault).astype(int)
    y_tune=dev.loc[tune_mask,'anomaly_type'].eq(fault).astype(int)
    ratio=(len(y_train)-y_train.sum())/max(y_train.sum(),1)
    models=[]; history=[]
    for seed in SPECIALIST_SEEDS:
        path=ARTIFACT_ROOT/f'{fault}_specialist_seed{seed}.cbm'
        model=CatBoostClassifier(iterations=1000,depth=7,learning_rate=.035,loss_function='Logloss',eval_metric='PRAUC',
            scale_pos_weight=min(math.sqrt(ratio),25),l2_leaf_reg=8,random_strength=.4,random_seed=seed,
            task_type='GPU',devices='0',verbose=100,od_type='Iter',od_wait=100,allow_writing_files=False)
        if REUSE_SAVED_MODELS and path.exists(): model.load_model(path)
        else:
            model.fit(dev.loc[train_mask,features],y_train,eval_set=(dev.loc[tune_mask,features],y_tune),use_best_model=True)
            model.save_model(path)
        best=model.get_best_iteration()
        models.append(model); history.append({'seed':seed,'best_iteration':int(best if best is not None else model.tree_count_-1)})
    specialist_models[fault]=models; specialist_history[fault]=history
    raw=np.mean([m.predict_proba(dev[features])[:,1] for m in models],axis=0)
    dev[f'{fault}_raw']=raw
print(specialist_history)


## 5. Rank-preserving Platt calibration for comparable rescue scores

In [ ]:
fit_mask=dev.dev_split.eq('block_may_jun')
calibrators={}

def fit_platt(raw,y):
    raw=np.clip(np.asarray(raw,float),1e-6,1-1e-6); logit=np.log(raw/(1-raw)).reshape(-1,1)
    model=LogisticRegression(C=1,max_iter=2000).fit(logit,np.asarray(y,int)); return model

def apply_platt(model,raw):
    raw=np.clip(np.asarray(raw,float),1e-6,1-1e-6); return model.predict_proba(np.log(raw/(1-raw)).reshape(-1,1))[:,1]

for fault in SPECIALIST_FEATURES:
    col=f'{fault}_raw'; y=dev.anomaly_type.eq(fault).astype(int)
    calibrators[fault]=fit_platt(dev.loc[fit_mask,col],y.loc[fit_mask])
    dev[f'{fault}_score']=apply_platt(calibrators[fault],dev[col])

base_calibrator=fit_platt(dev.loc[fit_mask,'cat_fault_mean'],dev.loc[fit_mask,'is_anomaly'])
dev['base_score']=apply_platt(base_calibrator,dev.cat_fault_mean)
dev['rescue_score']=dev[[f'{f}_score' for f in SPECIALIST_FEATURES]].max(axis=1)

rows=[]
for fault in SPECIALIST_FEATURES:
    for block in ['block_jul_sep','block_oct_dec']:
        m=dev.dev_split.eq(block); y=dev.anomaly_type.eq(fault).astype(int)
        rows.append({'fault':fault,'block':block,'auprc':average_precision_score(y[m],dev.loc[m,f'{fault}_score'])})
specialist_validation=pd.DataFrame(rows)
display(specialist_validation)


## 6. Hard rules and two-tier rescue policy

The high-precision CatBoost channel creates ordinary alerts. Frozen/bias/drift specialists may rescue an incident only after their evidence persists for multiple consecutive readings. Hard packet/timestamp/physical errors remain deterministic overrides.


In [ ]:
def add_hard_rules(frame):
    z=frame.copy()
    duplicate=z.duplicated(['station_id','emitted_timestamp_utc'],keep=False)
    timestamp=z.out_of_order_indicator.fillna(0).gt(0)
    physical=((z.temperature_value.notna()&~z.temperature_value.between(-60,60))|
              (z.pressure_value.notna()&~z.pressure_value.between(800,1100))|
              (z.humidity_value.notna()&~z.humidity_value.between(0,100)))
    z['hard_rule']=(duplicate|timestamp|physical)
    return z
dev=add_hard_rules(dev)

POLICY_BLOCKS=['block_may_jun','block_jul_sep','block_oct_dec']

def apply_two_tier(frame,base_start,base_continue,rescue_threshold,min_points):
    base=hysteresis(frame,'base_score',base_start,base_continue)
    rescue=persistent_signal(frame,'rescue_score',rescue_threshold,min_points)
    return base|rescue|frame.hard_rule

def search_robust_policy(frame):
    rows=[]
    for base_start in np.linspace(.30,.90,8):
      for rescue_threshold in [.50,.60,.70,.80,.90]:
       for min_points in [2,3,4]:
        block_metrics=[]
        for block in POLICY_BLOCKS:
            part=frame.loc[frame.dev_split.eq(block)].copy()
            part['pred']=apply_two_tier(part,base_start,max(0,base_start-.08),rescue_threshold,min_points)
            block_metrics.append((block,evaluate(part,'base_score','pred')))
        summary={'base_start':base_start,'base_continue':max(0,base_start-.08),
                 'rescue_threshold':rescue_threshold,'min_points':min_points}
        for block,m in block_metrics:
            for key in ['precision','recall','f1','auprc','event_precision','event_recall','event_f1',
                        'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']:
                summary[f'{block}_{key}']=m[key]
        summary['min_precision']=min(m['precision'] for _,m in block_metrics)
        summary['max_false_alarm']=max(m['false_alarm_episodes_per_station_day'] for _,m in block_metrics)
        summary['min_event_recall']=min(m['event_recall'] for _,m in block_metrics)
        summary['mean_event_f1']=np.mean([m['event_f1'] for _,m in block_metrics])
        summary['mean_point_f1']=np.mean([m['f1'] for _,m in block_metrics])
        rows.append(summary)
    frontier=pd.DataFrame(rows)
    feasible=frontier.loc[(frontier.min_precision>=.75)&(frontier.max_false_alarm<=.02)]
    if len(feasible):
        selected=feasible.sort_values(['min_event_recall','mean_event_f1','mean_point_f1'],ascending=False).iloc[0]
        status='constraints_met_all_blocks'
    else:
        frontier['violation']=np.maximum(0,.75-frontier.min_precision)/.75+np.maximum(0,frontier.max_false_alarm-.02)/.02
        selected=frontier.sort_values(['violation','min_event_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
        status='pareto_fallback'
    return selected,frontier,status

selected,frontier,POLICY_STATUS=search_robust_policy(dev)
frontier.to_csv(ARTIFACT_ROOT/'iteration2_policy_frontier.csv',index=False)
display(selected.to_frame('selected')); print(POLICY_STATUS)


## 7. Multi-block ablation and weak-fault recall

In [ ]:
def robust_base_policy(frame,score_col):
    candidates=[]
    for threshold in np.linspace(.10,.95,25):
        block_results=[]
        for block in POLICY_BLOCKS:
            part=frame.loc[frame.dev_split.eq(block)].copy()
            part['pred']=hysteresis(part,score_col,threshold,max(0,threshold-.08))
            block_results.append(evaluate(part,score_col,'pred'))
        candidates.append({'threshold':threshold,
            'min_precision':min(m['precision'] for m in block_results),
            'max_false_alarm':max(m['false_alarm_episodes_per_station_day'] for m in block_results),
            'min_event_recall':min(m['event_recall'] for m in block_results),
            'mean_event_f1':np.mean([m['event_f1'] for m in block_results])})
    feasible=[x for x in candidates if x['min_precision']>=.75 and x['max_false_alarm']<=.02]
    return max(feasible or candidates,key=lambda x:(x['min_event_recall'],x['mean_event_f1']))

rows=[]; combined=[]
base_policy=robust_base_policy(dev,'base_score')
print('Robust CatBoost base policy:',base_policy)
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['phase10_pred']=part.phase10_fault.ge(PHASE10_THRESHOLD)
    rows.append({'block':block,'variant':'Phase10 fixed',**evaluate(part,'phase10_fault','phase10_pred')})
    part['cat_pred']=hysteresis(part,'base_score',base_policy['threshold'],max(0,base_policy['threshold']-.08))
    rows.append({'block':block,'variant':'CatBoost base',**evaluate(part,'base_score','cat_pred')})
    part['rescue_pred']=apply_two_tier(part,selected.base_start,selected.base_continue,selected.rescue_threshold,int(selected.min_points))
    rows.append({'block':block,'variant':'CatBoost + weak-fault rescue',**evaluate(part,'base_score','rescue_pred')})
    combined.append(part)
ablation=pd.DataFrame(rows)
ablation.to_csv(ARTIFACT_ROOT/'iteration2_multiblock_ablation.csv',index=False)
display(ablation[['block','variant','precision','recall','f1','auprc','event_precision','event_recall','event_f1',
                  'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.rescue_pred
fault_recall=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall.to_csv(ARTIFACT_ROOT/'iteration2_fault_episode_recall.csv')
display(fault_recall)


## 8. Repair the weather channel by selecting the robust existing score

In [ ]:
weather_candidates=['phase10_weather','cat_weather_mean','cat_weather_median']
weather_rows=[]
for score_col in weather_candidates:
 for threshold in np.linspace(.02,.90,45):
    metrics=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)]; pred=part[score_col].ge(threshold)
        metrics.append({'block':block,'precision':precision_score(part.is_weather_event,pred,zero_division=0),
                        'recall':recall_score(part.is_weather_event,pred,zero_division=0),
                        'f1':f1_score(part.is_weather_event,pred,zero_division=0)})
    weather_rows.append({'score':score_col,'threshold':threshold,'min_f1':min(m['f1'] for m in metrics),
                         'mean_f1':np.mean([m['f1'] for m in metrics]),'blocks':metrics})
weather_frontier=pd.DataFrame(weather_rows)
weather_selected=weather_frontier.sort_values(['min_f1','mean_f1'],ascending=False).iloc[0]
weather_frontier.drop(columns='blocks').to_csv(ARTIFACT_ROOT/'iteration2_weather_frontier.csv',index=False)
display(weather_selected.to_frame('selected'))


## 9. Save Iteration 2 result package

In [ ]:
result={
 'iteration':'02_weak_fault_rescue','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':bool(UNLOCK_FINAL_TESTS),'tcn_promoted':False,'isotonic_fusion_promoted':False,
 'specialist_history':specialist_history,'specialist_validation':specialist_validation.to_dict('records'),
 'policy_status':POLICY_STATUS,'selected_policy':selected.to_dict(),'robust_base_policy':base_policy,
 'multiblock_ablation':ablation.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall':fault_recall.episode_recall.to_dict(),
 'weather_selected':{'score':weather_selected.score,'threshold':float(weather_selected.threshold),
                     'min_f1':float(weather_selected.min_f1),'mean_f1':float(weather_selected.mean_f1)},
}
(ARTIFACT_ROOT/'iteration2_result_block.json').write_text(json.dumps(result,indent=2,default=float))
joblib.dump({'specialist_calibrators':calibrators,'base_calibrator':base_calibrator,
             'specialist_features':SPECIALIST_FEATURES,'selected_policy':selected.to_dict(),
             'weather_score':weather_selected.score,'weather_threshold':float(weather_selected.threshold)},
            ARTIFACT_ROOT/'iteration2_policy.joblib')
print(json.dumps(result,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES — continue running; do not return these yet:')
for name in ['iteration2_result_block.json','iteration2_multiblock_ablation.csv','iteration2_fault_episode_recall.csv','iteration2_weather_frontier.csv']:
    print(ARTIFACT_ROOT/name)


## 10. Final-test seal

Iteration 2 intentionally contains no final-test scoring cell. Even if `UNLOCK_FINAL_TESTS` is changed accidentally, no 2024 file is loaded. After reviewing these outputs, we will either retain CatBoost alone or freeze the rescue policy, and only a separate finalization notebook will open the tests once.


## Historical checkpoint — continue running

Send the four files listed above. We will accept the rescue policy only if it improves weak-fault/event recall consistently across the three 2023 blocks while preserving precision and the 0.02 false-alarm budget. Otherwise CatBoost alone remains the winner.


# Iteration 3 controlled experiment

Everything above reconstructs Iteration 2 and reuses its saved models. The cells below write only to `iteration_03_frozen_communication`.


In [ ]:
ITER3_ROOT=DRIVE_ROOT/'experiments'/'iteration_03_frozen_communication'
ITER3_ROOT.mkdir(parents=True,exist_ok=True)
print('Iteration 3 artifacts:',ITER3_ROOT)


## 11. Sensor-specific frozen rules

Natural quantization differs strongly by sensor. Therefore a single frozen threshold is inappropriate. Candidate rules require both a constant-value run and disagreement from currently available neighbouring stations.


In [ ]:
FROZEN_CANDIDATES={
 'temperature':[None,(4,5.0),(8,4.0),(12,3.0)],
 'pressure':[None,(12,5.0),(16,3.0),(20,2.0)],
 'humidity':[None,(16,10.0),(12,10.0),(20,5.0)],
}

def sensor_frozen_rule(frame,sensor,config):
    if config is None: return pd.Series(False,index=frame.index)
    run_threshold,residual_threshold=config
    return (frame[f'{sensor}_frozen_run_length'].ge(run_threshold)&
            frame[f'neighbor_{sensor}_count'].ge(1)&
            frame[f'neighbor_{sensor}_residual'].abs().ge(residual_threshold))

def frozen_rule(frame,configs):
    result=pd.Series(False,index=frame.index)
    for sensor,config in configs.items(): result|=sensor_frozen_rule(frame,sensor,config)
    return result

def evaluate_frozen_configuration(configs):
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['iteration2_pred']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                               selected.rescue_threshold,int(selected.min_points))
        part['frozen_rule']=frozen_rule(part,configs)
        part['candidate_pred']=part.iteration2_pred|part.frozen_rule
        reference=evaluate(part,'base_score','iteration2_pred')
        metric=evaluate(part,'base_score','candidate_pred')
        rows.append({'block':block,**metric,
                     'event_f1_delta':metric['event_f1']-reference['event_f1'],
                     'point_f1_delta':metric['f1']-reference['f1']})
    valid_frozen=[row['per_fault_episode_recall'].get('frozen_sensor') for row in rows
                  if row['per_fault_episode_recall'].get('frozen_sensor') is not None]
    return {
        'temperature':str(configs['temperature']),'pressure':str(configs['pressure']),
        'humidity':str(configs['humidity']),'rows':rows,
        'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_event_recall':min(row['event_recall'] for row in rows),
        'mean_event_f1':float(np.mean([row['event_f1'] for row in rows])),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
        'min_frozen_recall':min(valid_frozen) if valid_frozen else 0.0,
        'mean_frozen_recall':float(np.mean(valid_frozen)) if valid_frozen else 0.0,
    }

candidate_rows=[]
for temperature in FROZEN_CANDIDATES['temperature']:
 for pressure in FROZEN_CANDIDATES['pressure']:
  for humidity in FROZEN_CANDIDATES['humidity']:
    candidate_rows.append(evaluate_frozen_configuration({
        'temperature':temperature,'pressure':pressure,'humidity':humidity}))

frontier=pd.DataFrame([{k:v for k,v in row.items() if k!='rows'} for row in candidate_rows])
baseline_frozen=float(frontier.loc[
    frontier[['temperature','pressure','humidity']].eq('None').all(axis=1),'mean_frozen_recall'].iloc[0])
feasible=frontier.loc[(frontier.min_precision>=.75)&(frontier.max_false_alarm<=.02)&
                      (frontier.min_event_f1_delta>=-.01)&(frontier.mean_event_f1_delta>=0)]
if len(feasible):
    frozen_selected=feasible.sort_values(
        ['min_frozen_recall','mean_frozen_recall','mean_event_f1_delta','min_event_recall'],ascending=False).iloc[0]
    FROZEN_STATUS='constraints_met_all_blocks'
else:
    frontier['violation']=np.maximum(0,.75-frontier.min_precision)/.75+np.maximum(0,frontier.max_false_alarm-.02)/.02
    frozen_selected=frontier.sort_values(['violation','min_frozen_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
    FROZEN_STATUS='pareto_fallback'

def parse_config(value):
    if value=='None': return None
    left,right=value.strip('()').split(','); return (int(left),float(right))

SELECTED_FROZEN_CONFIG={sensor:parse_config(frozen_selected[sensor]) for sensor in ['temperature','pressure','humidity']}
if float(frozen_selected.mean_frozen_recall)<=baseline_frozen:
    SELECTED_FROZEN_CONFIG={sensor:None for sensor in ['temperature','pressure','humidity']}
    FROZEN_STATUS='no_generalizable_gain_keep_iteration2'
frontier.to_csv(ITER3_ROOT/'iteration3_frozen_frontier.csv',index=False)
display(frozen_selected.to_frame('selected')); print(FROZEN_STATUS,SELECTED_FROZEN_CONFIG)


## 12. Clean contribution ablation

In [ ]:
rows=[]; combined=[]
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['cat_base']=hysteresis(part,'base_score',base_policy['threshold'],max(0,base_policy['threshold']-.08))
    part['cat_hard']=part.cat_base|part.hard_rule
    part['iteration2']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                      selected.rescue_threshold,int(selected.min_points))
    part['frozen_only']=part.cat_hard|frozen_rule(part,SELECTED_FROZEN_CONFIG)
    part['iteration3']=part.iteration2|frozen_rule(part,SELECTED_FROZEN_CONFIG)
    for variant,pred in [('CatBoost base','cat_base'),('CatBoost + hard rules','cat_hard'),
                         ('Iteration2 learned rescue','iteration2'),('CatBoost + frozen rule','frozen_only'),
                         ('Iteration3 combined','iteration3')]:
        rows.append({'block':block,'variant':variant,**evaluate(part,'base_score',pred)})
    combined.append(part)

ablation3=pd.DataFrame(rows)
ablation3.to_csv(ITER3_ROOT/'iteration3_multiblock_ablation.csv',index=False)
display(ablation3[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.iteration3
fault_recall3=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall3.to_csv(ITER3_ROOT/'iteration3_fault_episode_recall.csv')
display(fault_recall3)


## 13. Correct duplicate-packet evaluation through replay

The detector never reads `stream_action`. The simulator reads it and emits an identical packet twice. The operational rule then detects the second packet by a station/timestamp/value fingerprint. This mirrors the actual streaming engine.


In [ ]:
validation_full=load_table('validation')
replay_packets=[]
for _,row in validation_full.iterrows():
    packet={'station_id':row.station_id,'timestamp':row.emitted_timestamp_utc,
            'temperature':row.temperature_value,'pressure':row.pressure_value,'humidity':row.humidity_value,
            'episode_id':row.episode_id,'injected_duplicate':False}
    replay_packets.append(packet)
    if row.stream_action=='duplicate':
        duplicate=packet.copy(); duplicate['injected_duplicate']=True; replay_packets.append(duplicate)

def packet_fingerprint(packet):
    clean=lambda value: None if pd.isna(value) else value
    return (packet['station_id'],packet['timestamp'],clean(packet['temperature']),
            clean(packet['pressure']),clean(packet['humidity']))

seen=set(); tp=fp=fn=0; detected_episodes=set(); true_episodes=set()
for packet in replay_packets:
    fingerprint=packet_fingerprint(packet)
    prediction=fingerprint in seen; seen.add(fingerprint)
    truth=bool(packet['injected_duplicate'])
    tp+=int(prediction and truth); fp+=int(prediction and not truth); fn+=int(not prediction and truth)
    if truth and packet['episode_id']: true_episodes.add(packet['episode_id'])
    if prediction and truth and packet['episode_id']: detected_episodes.add(packet['episode_id'])

communication_result={
    'simulated_input_rows':int(len(validation_full)),
    'emitted_packets':int(len(replay_packets)),
    'duplicate_packet_tp':tp,'duplicate_packet_fp':fp,'duplicate_packet_fn':fn,
    'duplicate_packet_precision':tp/max(tp+fp,1),
    'duplicate_packet_recall':tp/max(tp+fn,1),
    'true_duplicate_episodes':len(true_episodes),
    'detected_duplicate_episodes':len(true_episodes&detected_episodes),
    'duplicate_episode_recall':len(true_episodes&detected_episodes)/max(len(true_episodes),1),
    'detector_inputs':['station_id','timestamp','temperature','pressure','humidity'],
    'stream_action_used_by_detector':False,
}
pd.DataFrame([communication_result]).to_csv(ITER3_ROOT/'iteration3_communication_replay.csv',index=False)
display(pd.Series(communication_result,name='replay'))


## 14. Save Iteration 3 result block

In [ ]:
iteration3_rows=ablation3.loc[ablation3.variant.eq('Iteration3 combined')]
result3={
 'iteration':'03_frozen_communication','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':False,'frozen_policy_status':FROZEN_STATUS,
 'selected_frozen_config':SELECTED_FROZEN_CONFIG,
 'iteration3_blocks':iteration3_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall':fault_recall3.episode_recall.to_dict(),
 'communication_replay':communication_result,
 'weather_policy_unchanged':{'score':weather_selected.score,'threshold':float(weather_selected.threshold)},
}
(ITER3_ROOT/'iteration3_result_block.json').write_text(json.dumps(result3,indent=2,default=float))
print(json.dumps(result3,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES — continue running; do not return these yet:')
for name in ['iteration3_result_block.json','iteration3_multiblock_ablation.csv',
             'iteration3_fault_episode_recall.csv','iteration3_communication_replay.csv']:
    print(ITER3_ROOT/name)


## Decision rule

Promote the frozen rule only if it raises frozen episode recall while all three blocks retain precision ≥75% and false-alarm episodes ≤0.02/station-day. Duplicate-packet replay must reach 100% episode recall. The final 2024 evaluation remains a separate, one-time notebook after this result is reviewed.


# Iteration 4 controlled experiment

Everything above reconstructs Iterations 2 and 3 from saved Drive checkpoints. The cells below write only to `iteration_04_causal_incident_state`.


In [ ]:
ITER4_ROOT=DRIVE_ROOT/'experiments'/'iteration_04_causal_incident_state'
ITER4_ROOT.mkdir(parents=True,exist_ok=True)
print('Iteration 4 artifacts:',ITER4_ROOT)


## 15. Causal incident-state controller

`grace_minutes` allows a live alert to remain active briefly after its latest strong or supporting observation. `support_threshold` is applied to the maximum calibrated CatBoost/specialist score and cannot start an incident. `max_duration_minutes` prevents an alert from remaining active indefinitely.


In [ ]:
def causal_incident_state(frame,trigger_col,grace_minutes,support_threshold,max_duration_minutes):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc')
        times=g.emitted_timestamp_utc.tolist()
        triggers=g[trigger_col].fillna(False).to_numpy(bool)
        support=np.maximum(g.base_score.fillna(0).to_numpy(float),g.rescue_score.fillna(0).to_numpy(float))
        out=np.zeros(len(g),bool); active=False; start=None; last_evidence=None
        for pos,(timestamp,trigger,score) in enumerate(zip(times,triggers,support)):
            if trigger:
                if not active: start=timestamp
                active=True; last_evidence=timestamp; out[pos]=True
                continue
            if not active: continue
            elapsed=(timestamp-start).total_seconds()/60
            since_evidence=(timestamp-last_evidence).total_seconds()/60
            if elapsed>max_duration_minutes:
                active=False; start=None; last_evidence=None; continue
            if score>=support_threshold:
                last_evidence=timestamp; out[pos]=True
            elif since_evidence<=grace_minutes:
                out[pos]=True
            else:
                active=False; start=None; last_evidence=None
        result.loc[g.index]=out
    return result

STATE_CANDIDATES={
    'grace_minutes':[0,60,180,360],
    'support_threshold':[.10,.20,.30,1.10],
    'max_duration_minutes':[720,1440],
}

STATE_REFERENCE={}
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3_trigger']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                              selected.rescue_threshold,int(selected.min_points))
    part['iteration3_trigger']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    STATE_REFERENCE[block]=evaluate(part,'base_score','iteration3_trigger')

def evaluate_state_configuration(grace_minutes,support_threshold,max_duration_minutes):
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['iteration3_trigger']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                                  selected.rescue_threshold,int(selected.min_points))
        part['iteration3_trigger']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
        part['candidate_pred']=causal_incident_state(part,'iteration3_trigger',grace_minutes,
                                                     support_threshold,max_duration_minutes)
        metric=evaluate(part,'base_score','candidate_pred'); reference=STATE_REFERENCE[block]
        rows.append({'block':block,**metric,
                     'point_f1_delta':metric['f1']-reference['f1'],
                     'event_f1_delta':metric['event_f1']-reference['event_f1']})
    return {
        'grace_minutes':grace_minutes,'support_threshold':support_threshold,
        'max_duration_minutes':max_duration_minutes,'rows':rows,
        'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_point_f1':min(row['f1'] for row in rows),
        'mean_point_f1':float(np.mean([row['f1'] for row in rows])),
        'min_point_f1_delta':min(row['point_f1_delta'] for row in rows),
        'mean_point_f1_delta':float(np.mean([row['point_f1_delta'] for row in rows])),
        'min_event_recall':min(row['event_recall'] for row in rows),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
    }

state_candidates=[]
for grace in STATE_CANDIDATES['grace_minutes']:
 for support in STATE_CANDIDATES['support_threshold']:
  for maximum in STATE_CANDIDATES['max_duration_minutes']:
    state_candidates.append(evaluate_state_configuration(grace,support,maximum))

state_frontier=pd.DataFrame([{k:v for k,v in row.items() if k!='rows'} for row in state_candidates])
feasible=state_frontier.loc[(state_frontier.min_precision>=.75)&
                            (state_frontier.max_false_alarm<=.02)&
                            (state_frontier.min_point_f1_delta>=0)&
                            (state_frontier.min_event_f1_delta>=-.01)&
                            (state_frontier.mean_event_f1_delta>=0)]
if len(feasible):
    state_selected=feasible.sort_values(
        ['min_point_f1','mean_point_f1','min_event_recall','mean_event_f1_delta'],ascending=False).iloc[0]
    STATE_STATUS='constraints_met_all_blocks'
    safe_selection=True
else:
    state_selected=state_frontier.sort_values(
        ['min_point_f1_delta','mean_point_f1_delta','mean_event_f1_delta'],ascending=False).iloc[0]
    STATE_STATUS='no_safe_candidate_keep_iteration3'
    safe_selection=False

if not safe_selection:
    SELECTED_STATE={'grace_minutes':0,'support_threshold':1.10,'max_duration_minutes':720}
elif float(state_selected.mean_point_f1_delta)<=0:
    STATE_STATUS='no_generalizable_gain_keep_iteration3'
    SELECTED_STATE={'grace_minutes':0,'support_threshold':1.10,'max_duration_minutes':720}
else:
    SELECTED_STATE={k:(int(state_selected[k]) if k!='support_threshold' else float(state_selected[k]))
                    for k in ['grace_minutes','support_threshold','max_duration_minutes']}

state_frontier.to_csv(ITER4_ROOT/'iteration4_state_frontier.csv',index=False)
display(state_selected.to_frame('selected')); print(STATE_STATUS,SELECTED_STATE)


## 16. Multiblock ablation and fault coverage

In [ ]:
rows=[]; combined=[]
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                      selected.rescue_threshold,int(selected.min_points))
    part['iteration3']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    part['iteration4']=causal_incident_state(part,'iteration3',**SELECTED_STATE)
    for variant,pred in [('Iteration3 trigger','iteration3'),('Iteration4 causal state','iteration4')]:
        rows.append({'block':block,'variant':variant,**evaluate(part,'base_score',pred)})
    combined.append(part)

ablation4=pd.DataFrame(rows)
ablation4.to_csv(ITER4_ROOT/'iteration4_multiblock_ablation.csv',index=False)
display(ablation4[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.iteration4
fault_recall4=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall4.to_csv(ITER4_ROOT/'iteration4_fault_episode_recall.csv')

fault_rows=combined.loc[combined.is_anomaly.eq(1)]
point_fault_recall4=(fault_rows.groupby('anomaly_type').pred.mean().rename('point_recall').sort_values().to_frame())
point_fault_recall4.to_csv(ITER4_ROOT/'iteration4_point_fault_recall.csv')
display(fault_recall4); display(point_fault_recall4)


## 17. Operational communication coverage

Duplicate packets are evaluated through replay, not static feature rows. Therefore the operational coverage table replaces the static duplicate value with the validated replay result while retaining both values for auditability.


In [ ]:
operational_fault_recall=fault_recall4.episode_recall.to_dict()
static_duplicate_recall=float(operational_fault_recall.get('duplicate_packet',0.0))
operational_fault_recall['duplicate_packet']=float(communication_result['duplicate_episode_recall'])
communication_coverage={
    'static_duplicate_recall_not_operational':static_duplicate_recall,
    'replay_duplicate_packet_precision':float(communication_result['duplicate_packet_precision']),
    'replay_duplicate_packet_recall':float(communication_result['duplicate_packet_recall']),
    'replay_duplicate_episode_recall':float(communication_result['duplicate_episode_recall']),
}
display(pd.Series(communication_coverage,name='communication'))


## 18. Save Iteration 4 result block

In [ ]:
iteration4_rows=ablation4.loc[ablation4.variant.eq('Iteration4 causal state')]
result4={
 'iteration':'04_causal_incident_state','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':False,'state_policy_status':STATE_STATUS,'selected_state':SELECTED_STATE,
 'iteration4_blocks':iteration4_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall_static':fault_recall4.episode_recall.to_dict(),
 'fault_episode_recall_operational':operational_fault_recall,
 'point_fault_recall':point_fault_recall4.point_recall.to_dict(),
 'communication_coverage':communication_coverage,
 'frozen_rule_promoted':any(value is not None for value in SELECTED_FROZEN_CONFIG.values()),
}
(ITER4_ROOT/'iteration4_result_block.json').write_text(json.dumps(result4,indent=2,default=float))
print(json.dumps(result4,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES — continue running; do not return these yet:')
for name in ['iteration4_result_block.json','iteration4_state_frontier.csv',
             'iteration4_multiblock_ablation.csv','iteration4_fault_episode_recall.csv',
             'iteration4_point_fault_recall.csv']:
    print(ITER4_ROOT/name)


## Decision rule

Promote the causal state only if every development block preserves at least 75% precision, at most 0.02 false-alarm episodes per station-day, non-decreasing point F1, no more than 0.01 event-F1 loss in any block, non-decreasing mean event F1, and a positive mean point-F1 gain. Otherwise keep the Iteration 3/2 detector unchanged.

This experiment changes alert persistence only. It does not claim to improve previously missed frozen, drift, or bias episodes, and it does not access the 2024 final tests.


# Iteration 5 controlled experiment

Everything above reconstructs Iterations 2–4 from the saved Drive checkpoints. Iteration 4 selected a zero-grace fallback, so the reference detector below is exactly the validated Iteration 3 trigger. These cells write only to `iteration_05_weak_fault_consensus`.


In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

ITER5_ROOT=DRIVE_ROOT/'experiments'/'iteration_05_weak_fault_consensus'
ITER5_ROOT.mkdir(parents=True,exist_ok=True)
assert UNLOCK_FINAL_TESTS is False, 'Iteration 5 must not open final tests.'

WEAK_TYPES=('frozen_sensor','bias','drift')
WEAK_FEATURES=list(FEATURES)
assert len(WEAK_FEATURES)==108
assert not ({'hour_sin','hour_cos','day_of_year_sin','day_of_year_cos','temperature_dewpoint_spread_c'}&set(WEAK_FEATURES))
print('Iteration 5 artifacts:',ITER5_ROOT)
print('Weak fault families:',WEAK_TYPES,'| features:',len(WEAK_FEATURES))


## 19. Episode-balanced weak-fault experts

The former per-fault specialists trained with row-level imbalance. A 100-reading drift incident could therefore influence fitting far more than a 5-reading frozen incident. Here every positive episode receives equal total weight before the usual class-imbalance adjustment. The model still sees only the permitted, causal feature pipeline.

A shared weak-fault target is intentionally used instead of nine fault/sensor neural networks: the 2022 training partition contains only 2 frozen-humidity episodes and 3 temperature-bias episodes, which is not enough support for a trustworthy separate model.


In [ ]:
def make_episode_balanced_weights(frame,positive_mask):
    # Each weak-fault episode receives equal total positive weight. Labels are not used at inference.
    positive=np.asarray(positive_mask,bool)
    weights=np.ones(len(frame),dtype=float)
    positions=np.flatnonzero(positive)
    assert len(positions)>0
    positives=frame.iloc[positions][['episode_id','row_id']].copy()
    episode_key=positives.episode_id.fillna('').astype(str)
    episode_key=episode_key.where(episode_key.ne(''),'single_row_'+positives.row_id.astype(str))
    episode_length=episode_key.value_counts()
    per_row=episode_key.map(lambda key:1.0/episode_length.loc[key]).to_numpy(float)
    # Preserve the original total positive mass, while distributing it evenly between episodes.
    per_row*=len(per_row)/max(per_row.sum(),1e-12)
    weights[positions]=per_row
    imbalance=(len(frame)-len(positions))/max(weights[positions].sum(),1e-12)
    weights[positions]*=min(math.sqrt(imbalance),15.0)
    audit={
        'positive_rows':int(len(positions)),
        'positive_episodes':int(episode_key.nunique()),
        'median_episode_rows':float(episode_length.median()),
        'max_episode_rows':int(episode_length.max()),
        'positive_weight_sum':float(weights[positions].sum()),
        'negative_weight_sum':float(weights[~positive].sum()),
    }
    return weights,audit

weak_train_mask=dev.dev_split.eq('train')
weak_tune_mask=dev.dev_split.eq('tune_model')
weak_y=dev.anomaly_type.isin(WEAK_TYPES).astype(int)
weak_train_y=weak_y.loc[weak_train_mask].to_numpy(int)
weak_tune_y=weak_y.loc[weak_tune_mask].to_numpy(int)
weak_train_weights,weak_weight_audit=make_episode_balanced_weights(dev.loc[weak_train_mask],weak_train_y.astype(bool))

assert weak_train_y.sum()>100 and weak_tune_y.sum()>30
print('Episode-balanced training audit:',weak_weight_audit)
print('Train positives:',int(weak_train_y.sum()),'Tune positives:',int(weak_tune_y.sum()))


In [ ]:
WEAK_SEEDS=[17,41,67]
weak_cat_models=[]; weak_lgb_models=[]; weak_model_history=[]
X_train=dev.loc[weak_train_mask,WEAK_FEATURES]
X_tune=dev.loc[weak_tune_mask,WEAK_FEATURES]

for seed in WEAK_SEEDS:
    cat_path=ITER5_ROOT/f'weak_union_catboost_seed{seed}.cbm'
    cat=CatBoostClassifier(
        iterations=1400,depth=7,learning_rate=.03,loss_function='Logloss',eval_metric='PRAUC',
        l2_leaf_reg=10,random_strength=.35,random_seed=seed,task_type='GPU',devices='0',
        verbose=150,od_type='Iter',od_wait=120,allow_writing_files=False,
    )
    if REUSE_SAVED_MODELS and cat_path.exists():
        cat.load_model(cat_path)
    else:
        cat.fit(X_train,weak_train_y,sample_weight=weak_train_weights,
                eval_set=(X_tune,weak_tune_y),use_best_model=True)
        cat.save_model(cat_path)
    weak_cat_models.append(cat)

    lgb_path=ITER5_ROOT/f'weak_union_lightgbm_seed{seed}.joblib'
    if REUSE_SAVED_MODELS and lgb_path.exists():
        lgb=joblib.load(lgb_path)
    else:
        lgb=LGBMClassifier(
            objective='binary',n_estimators=1600,learning_rate=.025,num_leaves=31,
            max_depth=-1,min_child_samples=40,subsample=.85,colsample_bytree=.85,
            reg_alpha=1.0,reg_lambda=10.0,random_state=seed,n_jobs=-1,
            verbosity=-1,force_col_wise=True,
        )
        lgb.fit(X_train,weak_train_y,sample_weight=weak_train_weights,
                eval_set=[(X_tune,weak_tune_y)],eval_metric='average_precision',
                callbacks=[early_stopping(120,verbose=False),log_evaluation(0)])
        joblib.dump(lgb,lgb_path)
    weak_lgb_models.append(lgb)

    cat_best=cat.get_best_iteration()
    lgb_best=getattr(lgb,'best_iteration_',None)
    weak_model_history.append({
        'seed':seed,
        'catboost_best_iteration':int(cat_best if cat_best is not None and cat_best>=0 else cat.tree_count_-1),
        'lightgbm_best_iteration':int(lgb_best if lgb_best else lgb.n_estimators),
    })

print('Weak expert histories:',weak_model_history)


In [ ]:
dev['weak_cat_score']=np.mean([model.predict_proba(dev[WEAK_FEATURES])[:,1] for model in weak_cat_models],axis=0)
dev['weak_lgb_score']=np.mean([model.predict_proba(dev[WEAK_FEATURES])[:,1] for model in weak_lgb_models],axis=0)
dev['weak_consensus_min']=np.minimum(dev.weak_cat_score,dev.weak_lgb_score)
dev['weak_consensus_geom']=np.sqrt(np.clip(dev.weak_cat_score,0,1)*np.clip(dev.weak_lgb_score,0,1))

weak_score_columns=['weak_cat_score','weak_lgb_score','weak_consensus_min','weak_consensus_geom']
weak_model_validation=[]
for split in ['tune_model',*POLICY_BLOCKS]:
    part=dev.loc[dev.dev_split.eq(split)]
    union_y=part.anomaly_type.isin(WEAK_TYPES).astype(int)
    for score_col in weak_score_columns:
        weak_model_validation.append({
            'split':split,'target':'weak_union','score':score_col,
            'auprc':float(average_precision_score(union_y,part[score_col])),
        })
    for fault in WEAK_TYPES:
        fault_y=part.anomaly_type.eq(fault).astype(int)
        for score_col in weak_score_columns:
            weak_model_validation.append({
                'split':split,'target':fault,'score':score_col,
                'auprc':float(average_precision_score(fault_y,part[score_col])),
            })
weak_model_validation=pd.DataFrame(weak_model_validation)
weak_model_validation.to_csv(ITER5_ROOT/'iteration5_weak_model_validation.csv',index=False)
display(weak_model_validation.pivot(index=['split','target'],columns='score',values='auprc').round(4))


## 20. Causal consensus gate and two-block policy selection

An Iteration 5 rescue can begin only when both model families agree, the score persists across consecutive emitted readings, and the observation is not classified by the existing weather channel as a likely genuine regional event. A temporal gap resets the persistence counter.

Thresholds are selected using May–September 2023. October–December 2023 is an internal confirmation block and must independently pass every safeguard. This makes the experiment stricter than reusing a single block for both calibration and reporting.


In [ ]:
def gap_aware_run_length(frame,score_col,threshold,max_gap_minutes):
    result=pd.Series(0,index=frame.index,dtype=int)
    for _,group in frame.groupby('station_id',sort=False):
        group=group.sort_values('emitted_timestamp_utc')
        scores=group[score_col].fillna(0).to_numpy(float)
        timestamps=group.emitted_timestamp_utc.tolist()
        runs=np.zeros(len(group),dtype=int); run=0; previous=None
        for position,(timestamp,score) in enumerate(zip(timestamps,scores)):
            if previous is not None and (timestamp-previous).total_seconds()/60>max_gap_minutes:
                run=0
            run=run+1 if score>=threshold else 0
            runs[position]=run
            previous=timestamp
        result.loc[group.index]=runs
    return result

def weak_episode_mean(metric):
    values=[metric['per_fault_episode_recall'].get(fault,np.nan) for fault in WEAK_TYPES]
    values=[value for value in values if not pd.isna(value)]
    return float(np.mean(values)) if values else float('nan')

WEAK_REFERENCE={}
dev['iteration3_reference']=False
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    baseline=apply_two_tier(part,selected.base_start,selected.base_continue,
                            selected.rescue_threshold,int(selected.min_points))
    baseline|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    dev.loc[part.index,'iteration3_reference']=baseline.to_numpy(bool)
    metrics=evaluate(part.assign(iteration3_reference=baseline),'base_score','iteration3_reference')
    WEAK_REFERENCE[block]=metrics

WEAK_SCORE_OPTIONS=['weak_consensus_min','weak_consensus_geom']
WEAK_THRESHOLDS=[.15,.25,.35,.45,.55,.65,.75]
WEAK_MIN_POINTS=[2,3,4,6]
WEAK_MAX_GAPS=[90,180]
WEAK_WEATHER_GUARDS=[True,False]
WEATHER_GUARD_THRESHOLD=float(weather_selected.threshold)
dev['weak_weather_safe']=dev.cat_weather_mean.fillna(0).lt(WEATHER_GUARD_THRESHOLD)

run_cache={}
for score_col in WEAK_SCORE_OPTIONS:
    for threshold in WEAK_THRESHOLDS:
        for max_gap in WEAK_MAX_GAPS:
            run_cache[(score_col,threshold,max_gap)]=gap_aware_run_length(dev,score_col,threshold,max_gap)

def evaluate_weak_policy(score_col,threshold,min_points,max_gap_minutes,weather_guard):
    rescue=run_cache[(score_col,threshold,max_gap_minutes)].ge(min_points)
    if weather_guard:
        rescue&=dev.weak_weather_safe
    candidate=dev.iteration3_reference|rescue
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['candidate']=candidate.loc[part.index].to_numpy(bool)
        metric=evaluate(part,'base_score','candidate')
        reference=WEAK_REFERENCE[block]
        weak_recall=weak_episode_mean(metric)
        base_weak_recall=weak_episode_mean(reference)
        rows.append({
            'block':block,**metric,
            'weak_episode_recall':weak_recall,
            'point_f1_delta':metric['f1']-reference['f1'],
            'event_f1_delta':metric['event_f1']-reference['event_f1'],
            'weak_episode_recall_delta':weak_recall-base_weak_recall,
        })
    summary={
        'score_col':score_col,'threshold':float(threshold),'min_points':int(min_points),
        'max_gap_minutes':int(max_gap_minutes),'weather_guard':bool(weather_guard),'rows':rows,
    }
    for scope,blocks in {'tune':['block_may_jun','block_jul_sep'],'all':POLICY_BLOCKS}.items():
        scoped=[row for row in rows if row['block'] in blocks]
        summary[f'{scope}_min_precision']=float(min(row['precision'] for row in scoped))
        summary[f'{scope}_max_false_alarm']=float(max(row['false_alarm_episodes_per_station_day'] for row in scoped))
        summary[f'{scope}_min_point_f1_delta']=float(min(row['point_f1_delta'] for row in scoped))
        summary[f'{scope}_mean_point_f1_delta']=float(np.mean([row['point_f1_delta'] for row in scoped]))
        summary[f'{scope}_min_event_f1_delta']=float(min(row['event_f1_delta'] for row in scoped))
        summary[f'{scope}_mean_event_f1_delta']=float(np.mean([row['event_f1_delta'] for row in scoped]))
        summary[f'{scope}_mean_weak_episode_recall_delta']=float(np.mean([row['weak_episode_recall_delta'] for row in scoped]))
    return summary

def passes_safety(summary,scope):
    return (
        summary[f'{scope}_min_precision']>=.75 and
        summary[f'{scope}_max_false_alarm']<=.02 and
        summary[f'{scope}_min_point_f1_delta']>=0 and
        summary[f'{scope}_min_event_f1_delta']>=-.01 and
        summary[f'{scope}_mean_event_f1_delta']>=0 and
        summary[f'{scope}_mean_point_f1_delta']>0 and
        summary[f'{scope}_mean_weak_episode_recall_delta']>0
    )

policy_candidates=[]
for score_col in WEAK_SCORE_OPTIONS:
    for threshold in WEAK_THRESHOLDS:
        for min_points in WEAK_MIN_POINTS:
            for max_gap in WEAK_MAX_GAPS:
                for weather_guard in WEAK_WEATHER_GUARDS:
                    policy_candidates.append(evaluate_weak_policy(
                        score_col,threshold,min_points,max_gap,weather_guard))

policy_frontier=pd.DataFrame([{key:value for key,value in row.items() if key!='rows'} for row in policy_candidates])
policy_frontier['passes_tune_gates']=policy_frontier.apply(lambda row:passes_safety(row.to_dict(),'tune'),axis=1)
policy_frontier['passes_all_gates']=policy_frontier.apply(lambda row:passes_safety(row.to_dict(),'all'),axis=1)
policy_frontier.to_csv(ITER5_ROOT/'iteration5_policy_frontier.csv',index=False)

tune_feasible=[candidate for candidate in policy_candidates if passes_safety(candidate,'tune')]
if tune_feasible:
    proposed=sorted(tune_feasible,key=lambda row:(
        row['tune_mean_weak_episode_recall_delta'],row['tune_mean_point_f1_delta'],
        row['tune_mean_event_f1_delta'],row['tune_min_precision']),reverse=True)[0]
    if passes_safety(proposed,'all'):
        ITER5_STATUS='constraints_met_with_confirmation'
        SELECTED_WEAK_POLICY={key:proposed[key] for key in ['score_col','threshold','min_points','max_gap_minutes','weather_guard']}
    else:
        ITER5_STATUS='failed_october_confirmation_keep_iteration3'
        SELECTED_WEAK_POLICY={'score_col':'weak_consensus_min','threshold':1.10,'min_points':999,'max_gap_minutes':90,'weather_guard':True}
else:
    proposed=sorted(policy_candidates,key=lambda row:(
        row['all_min_point_f1_delta'],row['all_mean_point_f1_delta'],row['all_mean_weak_episode_recall_delta']),reverse=True)[0]
    ITER5_STATUS='no_safe_development_gain_keep_iteration3'
    SELECTED_WEAK_POLICY={'score_col':'weak_consensus_min','threshold':1.10,'min_points':999,'max_gap_minutes':90,'weather_guard':True}

print('Tune-feasible candidates:',len(tune_feasible),'of',len(policy_candidates))
print('Iteration 5 status:',ITER5_STATUS)
display(pd.Series(SELECTED_WEAK_POLICY,name='selected_policy').to_frame())
display(policy_frontier.sort_values(['passes_all_gates','all_mean_weak_episode_recall_delta','all_mean_point_f1_delta'],ascending=False).head(12))


## 21. Full three-block ablation and fault coverage

In [ ]:
def materialize_weak_rescue(frame,policy):
    if int(policy['min_points'])>100:
        return pd.Series(False,index=frame.index)
    run=gap_aware_run_length(frame,policy['score_col'],float(policy['threshold']),int(policy['max_gap_minutes']))
    rescue=run.ge(int(policy['min_points']))
    if bool(policy['weather_guard']):
        rescue&=frame.cat_weather_mean.fillna(0).lt(WEATHER_GUARD_THRESHOLD)
    return rescue

rows=[]; combined=[]
dev['iteration5_rescue']=materialize_weak_rescue(dev,SELECTED_WEAK_POLICY)
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3']=part.iteration3_reference.to_numpy(bool)
    # Slice the full causal run so persistence before a block boundary is preserved exactly as evaluated.
    part['weak_rescue']=dev.loc[part.index,'iteration5_rescue'].to_numpy(bool)
    part['iteration5']=part.iteration3|part.weak_rescue
    for variant,pred_col in [('Iteration3 reference','iteration3'),('Iteration5 weak consensus','iteration5')]:
        metric=evaluate(part,'base_score',pred_col)
        metric['weak_episode_recall']=weak_episode_mean(metric)
        rows.append({'block':block,'variant':variant,**metric})
    combined.append(part)

ablation5=pd.DataFrame(rows)
ablation5.to_csv(ITER5_ROOT/'iteration5_multiblock_ablation.csv',index=False)
display(ablation5[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'weak_episode_recall','false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True)
combined['pred']=combined.iteration5
fault_recall5=(pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall')
               .sort_values().rename_axis('anomaly_type').reset_index())
fault_recall5.to_csv(ITER5_ROOT/'iteration5_fault_episode_recall.csv',index=False)
point_recall5=(combined.loc[combined.is_anomaly.eq(1)].groupby('anomaly_type').pred.mean()
               .rename('point_recall').sort_values().rename_axis('anomaly_type').reset_index())
point_recall5.to_csv(ITER5_ROOT/'iteration5_point_fault_recall.csv',index=False)
display(fault_recall5); display(point_recall5)


## 22. Save result package and stop rule

The selected policy is a deliberately impossible threshold whenever no candidate passes all safeguards. In that case the reported Iteration 5 rows are the unmodified Iteration 3 reference, not a disguised lower-quality model.


In [ ]:
iteration5_rows=ablation5.loc[ablation5.variant.eq('Iteration5 weak consensus')]
operational_fault_recall=dict(zip(fault_recall5.anomaly_type,fault_recall5.episode_recall))
operational_fault_recall['duplicate_packet']=float(communication_result['duplicate_episode_recall'])
result5={
    'iteration':'05_episode_balanced_weak_fault_consensus',
    'device':DEVICE,'gpu':torch.cuda.get_device_name(0),
    'final_tests_opened':False,
    'status':ITER5_STATUS,
    'source_baseline':'Iteration3 trigger; Iteration4 was an identical zero-grace fallback',
    'model_families':['episode-balanced CatBoost weak-union ensemble','episode-balanced LightGBM weak-union ensemble'],
    'feature_count':len(WEAK_FEATURES),'weak_fault_types':list(WEAK_TYPES),
    'episode_weight_audit':weak_weight_audit,
    'model_history':weak_model_history,
    'selected_policy':SELECTED_WEAK_POLICY,
    'policy_candidates':int(len(policy_candidates)),
    'tune_feasible_candidates':int(len(tune_feasible)),
    'promotion_gates':{
        'min_precision':.75,'max_false_alarm_episodes_per_station_day':.02,
        'minimum_point_f1_delta':0.0,'minimum_event_f1_delta':-.01,
        'minimum_mean_event_f1_delta':0.0,'positive_mean_point_f1_delta':True,
        'positive_mean_weak_episode_recall_delta':True,
        'october_december_confirmation_required':True,
    },
    'iteration5_blocks':iteration5_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
    'fault_episode_recall_static':dict(zip(fault_recall5.anomaly_type,fault_recall5.episode_recall)),
    'fault_episode_recall_operational':operational_fault_recall,
    'point_fault_recall':dict(zip(point_recall5.anomaly_type,point_recall5.point_recall)),
    'communication_coverage':{
        'replay_duplicate_packet_precision':float(communication_result['duplicate_packet_precision']),
        'replay_duplicate_packet_recall':float(communication_result['duplicate_packet_recall']),
        'replay_duplicate_episode_recall':float(communication_result['duplicate_episode_recall']),
    },
}
(ITER5_ROOT/'iteration5_result_block.json').write_text(json.dumps(result5,indent=2,default=float))
(ITER5_ROOT/'iteration5_feature_contract.json').write_text(json.dumps({
    'features':WEAK_FEATURES,'forbidden_features':['hour_sin','hour_cos','day_of_year_sin','day_of_year_cos','temperature_dewpoint_spread_c'],
    'causal':True,'detector_inputs':['temperature','pressure','relative_humidity']
},indent=2))
print(json.dumps(result5,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES — continue running; do not return these yet:')
for name in ['iteration5_result_block.json','iteration5_policy_frontier.csv','iteration5_multiblock_ablation.csv',
             'iteration5_fault_episode_recall.csv','iteration5_point_fault_recall.csv','iteration5_weak_model_validation.csv']:
    print(ITER5_ROOT/name)


## Historical Iteration 5 decision — continue to Iteration 6

Promote Iteration 5 only when a policy selected without October–December labels also passes October–December confirmation and every one of these all-block requirements:

- point precision at least 75%;
- false-alert episodes at most 0.02 per station-day;
- no point-F1 regression in any block;
- no event-F1 regression larger than one percentage point in any block;
- positive mean point-F1 gain;
- positive mean weak-fault episode-recall gain.

Otherwise keep Iteration 3/2 unchanged. Regardless of the result, do not open the previously inspected 2024 benchmark. The next trustworthy promotion step is a newly created blind time/station benchmark.


# Iteration 6 controlled development phase

No 2024 or 2025 final benchmark is opened. Selection uses discovery stations in May–September 2023. October–December and four pseudo-unseen development stations are confirmation sets only.


In [ ]:
ITER6_ROOT=DRIVE_ROOT/'experiments'/'iteration_06_communication_weather_transfer'
ITER6_ROOT.mkdir(parents=True,exist_ok=True)
assert UNLOCK_FINAL_TESTS is False,'Iteration 6 must keep former final tests locked.'
assert 'blind_2025' not in str(ITER6_ROOT).lower()

station_contract=(dev[['station_id','cluster']].drop_duplicates()
                  .sort_values(['cluster','station_id']).reset_index(drop=True))
PSEUDO_HOLDOUT_STATIONS=(station_contract.groupby('cluster',sort=True).station_id.last().astype(str).tolist())
DISCOVERY_STATIONS=sorted(set(dev.station_id.astype(str))-set(PSEUDO_HOLDOUT_STATIONS))
assert len(PSEUDO_HOLDOUT_STATIONS)==4 and len(DISCOVERY_STATIONS)==16

dev['iteration5_reference']=dev.iteration3_reference|dev.iteration5_rescue
print('Iteration 6 root:',ITER6_ROOT)
print('Pseudo-unseen stations:',PSEUDO_HOLDOUT_STATIONS)
display(station_contract.assign(pseudo_unseen=station_contract.station_id.astype(str).isin(PSEUDO_HOLDOUT_STATIONS)))


## 23. Communication-gap identifiability audit

An archive can omit reports for many reasons that are not labelled sensor faults. A gap-only detector cannot safely call every long silence a station failure unless the source supplies an expected heartbeat/cadence contract.

The detector below learns cadence statistics from 2022 arrival timestamps only. `stream_action` is used exclusively by the simulator/evaluator to omit injected packets and create ground truth; it is never an inference feature.


In [ ]:
def build_cadence_profiles(profile_frame):
    profiles={}
    emitted=(profile_frame.loc[profile_frame.stream_action.ne('drop')]
             .drop_duplicates(['station_id','emitted_timestamp_utc']))
    for station,group in emitted.groupby('station_id',sort=False):
        delta=(group.sort_values('emitted_timestamp_utc').emitted_timestamp_utc.diff()
               .dt.total_seconds().div(60))
        plausible=delta[(delta>0)&(delta<=180)].round().astype(int)
        cadence=float(plausible.mode().iloc[0]) if len(plausible) else 60.0
        all_ratio=(delta[delta>0]/max(cadence,1)).clip(upper=1000)
        profiles[str(station)]={
            'cadence_minutes':cadence,
            'training_regularity':float((abs(plausible-cadence)<=max(5,.2*cadence)).mean()),
            'training_gap_ratio_q99':float(all_ratio.quantile(.99)),
            'training_gap_ratio_q999':float(all_ratio.quantile(.999)),
        }
    return profiles

def build_gap_events(profile_frame,replay_frame):
    profiles=build_cadence_profiles(profile_frame)
    rows=[]
    for station,group in replay_frame.groupby('station_id',sort=False):
        station=str(station); group=group.sort_values('emitted_timestamp_utc')
        dropped=group.loc[group.stream_action.eq('drop'),['emitted_timestamp_utc','episode_id']].copy()
        observed=(group.loc[group.stream_action.ne('drop')]
                  .drop_duplicates('emitted_timestamp_utc').sort_values('emitted_timestamp_utc'))
        events=pd.DataFrame({'current':observed.emitted_timestamp_utc})
        events['previous']=events.current.shift()
        events=events.dropna().copy()
        events['gap_minutes']=(events.current-events.previous).dt.total_seconds()/60
        cadence=profiles[station]['cadence_minutes']
        events['cadence_minutes']=cadence
        events['gap_ratio']=events.gap_minutes/max(cadence,1)
        regular=(abs(events.gap_minutes-cadence)<=max(5,.2*cadence)).astype(float)
        events['recent_regularity']=regular.shift().rolling(48,min_periods=12).mean().fillna(0)
        events['training_regularity']=profiles[station]['training_regularity']
        events['training_gap_ratio_q99']=profiles[station]['training_gap_ratio_q99']
        events['training_gap_ratio_q999']=profiles[station]['training_gap_ratio_q999']
        events['station_id']=station
        timestamp=events.current
        events['dev_split']=np.select(
            [timestamp<'2023-05-01',timestamp<'2023-07-01',timestamp<'2023-10-01'],
            ['tune_model','block_may_jun','block_jul_sep'],default='block_oct_dec')

        drop_times=dropped.emitted_timestamp_utc.to_numpy(dtype='datetime64[ns]')
        starts=events.previous.to_numpy(dtype='datetime64[ns]')
        ends=events.current.to_numpy(dtype='datetime64[ns]')
        left=np.searchsorted(drop_times,starts,side='right')
        right=np.searchsorted(drop_times,ends,side='left')
        events['is_injected_dropout_gap']=right>left
        rows.append(events)
    return pd.concat(rows,ignore_index=True),profiles

# Colab may retain the reconstructed `dev` table while releasing the two source
# tables after an interrupted or out-of-order run. Reload only those lightweight
# inputs when necessary; no model is retrained and no final/blind file is opened.
if 'train' not in globals() or not isinstance(train,pd.DataFrame):
    train=load_table('train')
    train['dev_split']='train'
if 'validation' not in globals() or not isinstance(validation,pd.DataFrame):
    validation=load_table('validation')
    validation_timestamp=validation.emitted_timestamp_utc
    validation['dev_split']=np.select(
        [validation_timestamp<'2023-05-01',validation_timestamp<'2023-07-01',
         validation_timestamp<'2023-10-01'],
        ['tune_model','block_may_jun','block_jul_sep'],default='block_oct_dec')

gap_events6,cadence_profiles6=build_gap_events(train,validation)
print('Gap events:',len(gap_events6),'| injected dropout gaps:',int(gap_events6.is_injected_dropout_gap.sum()))
display(pd.DataFrame(cadence_profiles6).T.describe().T)


In [ ]:
def gap_policy_metrics(events,prediction):
    truth=events.is_injected_dropout_gap.to_numpy(bool); pred=np.asarray(prediction,bool)
    tp=int((truth&pred).sum()); fp=int((~truth&pred).sum()); fn=int((truth&~pred).sum())
    precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)
    first=events.previous.min(); last=events.current.max()
    station_days=max((last-first).total_seconds()/86400,1)*events.station_id.nunique()
    return {'tp':tp,'fp':fp,'fn':fn,'precision':precision,'recall':recall,
            'f1':2*precision*recall/max(precision+recall,1e-12),
            'false_gap_alerts_per_station_day':fp/max(station_days,1e-12)}

gap_selection=gap_events6.loc[
    gap_events6.station_id.astype(str).isin(DISCOVERY_STATIONS)&
    gap_events6.dev_split.isin(['block_may_jun','block_jul_sep'])].copy()
gap_frontier=[]
for ratio_threshold in [2.5,3,4,5,6,8,10,12,15,20,30,40]:
  for recent_regularity in [.50,.70,.80,.90,.95,.98]:
    for historical_multiplier in [0,1,1.5,2,3]:
        historical_floor=(gap_selection.training_gap_ratio_q999*historical_multiplier
                          if historical_multiplier else 0)
        threshold=np.maximum(ratio_threshold,historical_floor)
        prediction=(gap_selection.gap_ratio>=threshold)&(gap_selection.recent_regularity>=recent_regularity)
        gap_frontier.append({
            'ratio_threshold':ratio_threshold,'recent_regularity':recent_regularity,
            'historical_q999_multiplier':historical_multiplier,
            **gap_policy_metrics(gap_selection,prediction),
        })
gap_frontier=pd.DataFrame(gap_frontier).sort_values(['f1','precision','recall'],ascending=False)
gap_frontier.to_csv(ITER6_ROOT/'iteration6_communication_frontier.csv',index=False)

safe_gap_candidates=gap_frontier.loc[
    (gap_frontier.precision>=.80)&(gap_frontier.recall>=.80)&
    (gap_frontier.false_gap_alerts_per_station_day<=.02)]
gap_confirmation=[]
if len(safe_gap_candidates):
    proposed_gap=safe_gap_candidates.iloc[0].to_dict()
    for scope,mask in {
        'oct_dec_discovery':gap_events6.station_id.astype(str).isin(DISCOVERY_STATIONS)&gap_events6.dev_split.eq('block_oct_dec'),
        'pseudo_unseen_all_2023':gap_events6.station_id.astype(str).isin(PSEUDO_HOLDOUT_STATIONS)&gap_events6.dev_split.isin(POLICY_BLOCKS),
    }.items():
        part=gap_events6.loc[mask].copy()
        floor=(part.training_gap_ratio_q999*proposed_gap['historical_q999_multiplier']
               if proposed_gap['historical_q999_multiplier'] else 0)
        threshold=np.maximum(proposed_gap['ratio_threshold'],floor)
        pred=(part.gap_ratio>=threshold)&(part.recent_regularity>=proposed_gap['recent_regularity'])
        gap_confirmation.append({'scope':scope,**gap_policy_metrics(part,pred)})
    gap_confirmation_pass=all(
        row['precision']>=.80 and row['recall']>=.80 and row['false_gap_alerts_per_station_day']<=.02
        for row in gap_confirmation)
else:
    proposed_gap=None; gap_confirmation_pass=False

if proposed_gap and gap_confirmation_pass:
    GAP_POLICY_STATUS='automatic_archive_gap_policy_supported_with_confirmation'
    SELECTED_GAP_POLICY=proposed_gap
else:
    GAP_POLICY_STATUS='heartbeat_contract_required_archive_gaps_advisory_only'
    SELECTED_GAP_POLICY={
        'automatic_fault_alert_without_contract':False,
        'unknown_cadence_output':'unverified_data_gap_advisory',
        'strict_mode_requirement':'adapter supplies expected cadence and heartbeat SLA',
        'duplicate_packet_detection_remains_automatic':True,
    }

gap_confirmation=pd.DataFrame(gap_confirmation)
gap_confirmation.to_csv(ITER6_ROOT/'iteration6_communication_confirmation.csv',index=False)
print('Communication policy:',GAP_POLICY_STATUS)
display(gap_frontier.head(15)); display(gap_confirmation)
display(pd.Series(SELECTED_GAP_POLICY,name='selected').to_frame())


### Communication interpretation

If the safe frontier is empty, Iteration 6 does not hide the failure by selecting an extreme threshold. It changes the operational contract:

- duplicate packets remain automatic because their fingerprint has deterministic evidence;
- a configured AWS heartbeat may generate automatic dropout incidents;
- an unknown-cadence archive/live source generates an advisory data-gap status, not a sensor-fault maintenance alert.

This prevents thousands of unsupported fault claims while preserving strict dropout detection for sources that actually promise periodic delivery.


## 24. Station-invariant weather experts

Absolute temperature or pressure distributions can identify a station rather than weather coherence. The challenger therefore excludes raw values, raw lags, calendar encodings and dew-point-derived features. It uses causal rates, robust residuals, slopes, CUSUM and neighbour/regional agreement.


In [ ]:
WEATHER_INV_TOKENS=(
    'neighbor_','regional_','robust_z','ewma_residual','rate_per_hour','_delta1',
    '_slope_','cusum_','monotonic_run','rolling_mad','climatology_residual',
    'agreement_fraction','missing','gap_ratio','out_of_order_indicator',
)
WEATHER_INV_FEATURES=[feature for feature in FEATURES if any(token in feature for token in WEATHER_INV_TOKENS)]
FORBIDDEN_ABSOLUTE={
    'temperature_value','pressure_value','humidity_value','temperature_lag1','pressure_lag1','humidity_lag1',
    'temperature_rolling_median_24h','pressure_rolling_median_24h','humidity_rolling_median_24h',
}
WEATHER_INV_FEATURES=[feature for feature in WEATHER_INV_FEATURES if feature not in FORBIDDEN_ABSOLUTE]
assert len(WEATHER_INV_FEATURES)>=45
assert not (FORBIDDEN_ABSOLUTE&set(WEATHER_INV_FEATURES))
assert not ({'hour_sin','hour_cos','day_of_year_sin','day_of_year_cos','temperature_dewpoint_spread_c'}&set(WEATHER_INV_FEATURES))

def station_episode_balanced_weights(frame,target):
    target=np.asarray(target,bool); station=frame.station_id.astype(str)
    counts=station.value_counts(); weights=station.map(lambda value:1.0/counts.loc[value]).to_numpy(float)
    weights*=len(weights)/max(weights.sum(),1e-12)
    positions=np.flatnonzero(target)
    if len(positions):
        positive=frame.iloc[positions][['episode_id','row_id']].copy()
        key=positive.episode_id.fillna('').astype(str)
        key=key.where(key.ne(''),'single_'+positive.row_id.astype(str))
        episode_length=key.value_counts()
        episode_weight=key.map(lambda value:1.0/episode_length.loc[value]).to_numpy(float)
        episode_weight*=len(episode_weight)/max(episode_weight.sum(),1e-12)
        positive=positive.assign(station=station.iloc[positions].to_numpy())
        station_positive=positive.station.value_counts()
        station_weight=positive.station.map(lambda value:1.0/station_positive.loc[value]).to_numpy(float)
        station_weight*=len(station_weight)/max(station_weight.sum(),1e-12)
        combined=np.sqrt(episode_weight*station_weight)
        imbalance=(len(frame)-len(positions))/max(len(positions),1)
        weights[positions]=combined*min(math.sqrt(imbalance),20.0)
    return weights

discovery=dev.station_id.astype(str).isin(DISCOVERY_STATIONS)&dev.available_to_detector.eq(1)
pseudo=dev.station_id.astype(str).isin(PSEUDO_HOLDOUT_STATIONS)&dev.available_to_detector.eq(1)
weather_train=discovery&dev.dev_split.eq('train')
weather_tune=discovery&dev.dev_split.eq('tune_model')
weather_y=dev.is_weather_event.astype(int)
weather_weights=station_episode_balanced_weights(dev.loc[weather_train],weather_y.loc[weather_train].to_numpy(bool))

print('Weather invariant features:',len(WEATHER_INV_FEATURES))
print('Discovery train weather rows:',int(weather_y.loc[weather_train].sum()),
      '| tune:',int(weather_y.loc[weather_tune].sum()),
      '| pseudo-unseen validation:',int(weather_y.loc[pseudo&~dev.dev_split.eq('train')].sum()))


In [ ]:
WEATHER6_SEEDS=[17,41,67]
weather6_cat_models=[]; weather6_lgb_models=[]; weather6_history=[]
for seed in WEATHER6_SEEDS:
    cat_path=ITER6_ROOT/f'weather_invariant_catboost_seed{seed}.cbm'
    cat=CatBoostClassifier(
        iterations=1400,depth=7,learning_rate=.03,loss_function='Logloss',eval_metric='PRAUC',
        l2_leaf_reg=12,random_strength=.5,random_seed=seed,task_type='GPU',devices='0',
        verbose=150,od_type='Iter',od_wait=120,allow_writing_files=False,
    )
    if REUSE_SAVED_MODELS and cat_path.exists(): cat.load_model(cat_path)
    else:
        cat.fit(dev.loc[weather_train,WEATHER_INV_FEATURES],weather_y.loc[weather_train],
                sample_weight=weather_weights,
                eval_set=(dev.loc[weather_tune,WEATHER_INV_FEATURES],weather_y.loc[weather_tune]),
                use_best_model=True)
        cat.save_model(cat_path)
    weather6_cat_models.append(cat)

    lgb_path=ITER6_ROOT/f'weather_invariant_lightgbm_seed{seed}.joblib'
    if REUSE_SAVED_MODELS and lgb_path.exists(): lgb=joblib.load(lgb_path)
    else:
        lgb=LGBMClassifier(
            objective='binary',n_estimators=1800,learning_rate=.02,num_leaves=31,
            min_child_samples=50,subsample=.85,colsample_bytree=.80,
            reg_alpha=2.0,reg_lambda=15.0,random_state=seed,n_jobs=-1,
            verbosity=-1,force_col_wise=True,
        )
        lgb.fit(dev.loc[weather_train,WEATHER_INV_FEATURES],weather_y.loc[weather_train],
                sample_weight=weather_weights,
                eval_set=[(dev.loc[weather_tune,WEATHER_INV_FEATURES],weather_y.loc[weather_tune])],
                eval_metric='average_precision',callbacks=[early_stopping(120,verbose=False),log_evaluation(0)])
        joblib.dump(lgb,lgb_path)
    weather6_lgb_models.append(lgb)
    weather6_history.append({'seed':seed,'cat_trees':int(cat.tree_count_),
                             'lgb_iteration':int(getattr(lgb,'best_iteration_',0) or lgb.n_estimators)})

dev['weather6_cat']=np.mean([model.predict_proba(dev[WEATHER_INV_FEATURES])[:,1]
                             for model in weather6_cat_models],axis=0)
dev['weather6_lgb']=np.mean([model.predict_proba(dev[WEATHER_INV_FEATURES])[:,1]
                             for model in weather6_lgb_models],axis=0)
dev['weather6_mean']=(dev.weather6_cat+dev.weather6_lgb)/2
dev['weather6_min']=np.minimum(dev.weather6_cat,dev.weather6_lgb)
dev['weather6_geom']=np.sqrt(np.clip(dev.weather6_cat,0,1)*np.clip(dev.weather6_lgb,0,1))
print(weather6_history)


In [ ]:
def weather_quality(frame,pred_col,score_col):
    truth=frame.is_weather_event.astype(int); pred=frame[pred_col].astype(bool)
    precision=precision_score(truth,pred,zero_division=0); recall=recall_score(truth,pred,zero_division=0)
    station_f1=[]
    for _,group in frame.groupby('station_id'):
        if group.is_weather_event.sum()>0:
            station_f1.append(f1_score(group.is_weather_event,group[pred_col],zero_division=0))
    return {
        'weather_precision':float(precision),'weather_recall':float(recall),
        'weather_f1':float(f1_score(truth,pred,zero_division=0)),
        'weather_auprc':float(average_precision_score(truth,frame[score_col])),
        'positive_station_macro_f1':float(np.mean(station_f1)) if station_f1 else 0.0,
        'fault_to_weather_rate':float(pred.loc[frame.is_anomaly.eq(1)].mean()) if frame.is_anomaly.sum() else 0.0,
    }

WEATHER6_DISCOVERY_BLOCKS=['block_may_jun','block_jul_sep']
weather6_candidates=[]
for score_col in ['weather6_mean','weather6_min','weather6_geom']:
  for threshold in np.linspace(.05,.80,31):
    rows=[]
    for block in WEATHER6_DISCOVERY_BLOCKS:
        part=dev.loc[discovery&dev.dev_split.eq(block)].copy()
        part['current_pred']=part.cat_weather_mean.ge(float(WEATHER_GUARD_THRESHOLD))
        part['candidate_pred']=part[score_col].ge(threshold)
        current=weather_quality(part,'current_pred','cat_weather_mean')
        candidate=weather_quality(part,'candidate_pred',score_col)
        rows.append({'block':block,**candidate,'f1_delta':candidate['weather_f1']-current['weather_f1'],
                     'macro_delta':candidate['positive_station_macro_f1']-current['positive_station_macro_f1']})
    weather6_candidates.append({
        'score_col':score_col,'threshold':float(threshold),'rows':rows,
        'min_f1_delta':float(min(row['f1_delta'] for row in rows)),
        'mean_f1_delta':float(np.mean([row['f1_delta'] for row in rows])),
        'min_macro_delta':float(min(row['macro_delta'] for row in rows)),
        'max_fault_to_weather':float(max(row['fault_to_weather_rate'] for row in rows)),
        'mean_weather_f1':float(np.mean([row['weather_f1'] for row in rows])),
    })

weather6_frontier=pd.DataFrame([{key:value for key,value in row.items() if key!='rows'}
                                for row in weather6_candidates])
weather6_frontier['passes_discovery']=(
    (weather6_frontier.min_f1_delta>=0)&(weather6_frontier.mean_f1_delta>0)&
    (weather6_frontier.min_macro_delta>=-.02)&(weather6_frontier.max_fault_to_weather<=.01))
weather6_frontier.to_csv(ITER6_ROOT/'iteration6_weather_policy_frontier.csv',index=False)

feasible_weather=[row for row in weather6_candidates if (
    row['min_f1_delta']>=0 and row['mean_f1_delta']>0 and row['min_macro_delta']>=-.02 and
    row['max_fault_to_weather']<=.01)]
proposed_weather=(sorted(feasible_weather,key=lambda row:(row['mean_weather_f1'],row['mean_f1_delta'],
                                                          row['min_macro_delta']),reverse=True)[0]
                  if feasible_weather else None)

def compare_weather_scope(mask,score_col,threshold,scope):
    part=dev.loc[mask].copy()
    part['current_pred']=part.cat_weather_mean.ge(float(WEATHER_GUARD_THRESHOLD))
    part['candidate_pred']=part[score_col].ge(float(threshold))
    current=weather_quality(part,'current_pred','cat_weather_mean')
    candidate=weather_quality(part,'candidate_pred',score_col)
    return {'scope':scope,**{f'current_{k}':v for k,v in current.items()},
            **{f'candidate_{k}':v for k,v in candidate.items()},
            'f1_delta':candidate['weather_f1']-current['weather_f1'],
            'macro_delta':candidate['positive_station_macro_f1']-current['positive_station_macro_f1']}

weather6_confirmation=[]
if proposed_weather:
    score_col=proposed_weather['score_col']; threshold=proposed_weather['threshold']
    weather6_confirmation.append(compare_weather_scope(
        discovery&dev.dev_split.eq('block_oct_dec'),score_col,threshold,'oct_dec_discovery'))
    weather6_confirmation.append(compare_weather_scope(
        pseudo&dev.dev_split.isin(POLICY_BLOCKS),score_col,threshold,'pseudo_unseen_all_2023'))
    confirmation_pass=all(
        row['f1_delta']>=0 and row['macro_delta']>=-.02 and row['candidate_fault_to_weather_rate']<=.01
        for row in weather6_confirmation)
else:
    confirmation_pass=False

if proposed_weather and confirmation_pass:
    WEATHER6_STATUS='station_invariant_weather_confirmed'
    SELECTED_WEATHER6={'score_col':proposed_weather['score_col'],'threshold':proposed_weather['threshold']}
else:
    WEATHER6_STATUS='no_confirmed_weather_gain_keep_iteration5_weather'
    SELECTED_WEATHER6={'score_col':'cat_weather_mean','threshold':float(WEATHER_GUARD_THRESHOLD)}

weather6_confirmation=pd.DataFrame(weather6_confirmation)
weather6_confirmation.to_csv(ITER6_ROOT/'iteration6_weather_confirmation.csv',index=False)
dev['weather6_prediction']=dev[SELECTED_WEATHER6['score_col']].ge(float(SELECTED_WEATHER6['threshold']))
print('Weather status:',WEATHER6_STATUS,SELECTED_WEATHER6)
display(weather6_frontier.sort_values(['passes_discovery','mean_weather_f1'],ascending=False).head(15))
display(weather6_confirmation)


## 25. Station-balanced weak-fault transfer challenger

Iteration 5 used all 108 features and improved only known-station future time. This challenger removes absolute station-identifying values, balances positive episodes and stations, and excludes four pseudo-unseen stations from fitting and policy selection.


In [ ]:
WEAK6_TOKENS=(
    'neighbor_','regional_','robust_z','ewma_residual','rate_per_hour','_delta1','_slope_',
    'cusum_','monotonic_run','frozen_run_length','rolling_mad','climatology_residual',
    'missing','gap_ratio','out_of_order_indicator','time_since_previous_minutes',
)
WEAK6_FEATURES=[feature for feature in FEATURES if any(token in feature for token in WEAK6_TOKENS)]
WEAK6_FEATURES=[feature for feature in WEAK6_FEATURES if feature not in FORBIDDEN_ABSOLUTE]
assert len(WEAK6_FEATURES)>=50
assert not (FORBIDDEN_ABSOLUTE&set(WEAK6_FEATURES))

weak6_y=dev.anomaly_type.isin(WEAK_TYPES).astype(int)
weak6_train=discovery&dev.dev_split.eq('train')
weak6_tune=discovery&dev.dev_split.eq('tune_model')
weak6_weights=station_episode_balanced_weights(dev.loc[weak6_train],weak6_y.loc[weak6_train].to_numpy(bool))
print('Weak transfer features:',len(WEAK6_FEATURES),'| train positives:',int(weak6_y.loc[weak6_train].sum()))


In [ ]:
WEAK6_SEEDS=[17,41,67]
weak6_cat_models=[]; weak6_lgb_models=[]; weak6_history=[]
for seed in WEAK6_SEEDS:
    cat_path=ITER6_ROOT/f'weak_transfer_catboost_seed{seed}.cbm'
    cat=CatBoostClassifier(
        iterations=1500,depth=7,learning_rate=.025,loss_function='Logloss',eval_metric='PRAUC',
        l2_leaf_reg=15,random_strength=.55,random_seed=seed,task_type='GPU',devices='0',
        verbose=150,od_type='Iter',od_wait=130,allow_writing_files=False,
    )
    if REUSE_SAVED_MODELS and cat_path.exists(): cat.load_model(cat_path)
    else:
        cat.fit(dev.loc[weak6_train,WEAK6_FEATURES],weak6_y.loc[weak6_train],sample_weight=weak6_weights,
                eval_set=(dev.loc[weak6_tune,WEAK6_FEATURES],weak6_y.loc[weak6_tune]),use_best_model=True)
        cat.save_model(cat_path)
    weak6_cat_models.append(cat)

    lgb_path=ITER6_ROOT/f'weak_transfer_lightgbm_seed{seed}.joblib'
    if REUSE_SAVED_MODELS and lgb_path.exists(): lgb=joblib.load(lgb_path)
    else:
        lgb=LGBMClassifier(
            objective='binary',n_estimators=1800,learning_rate=.02,num_leaves=31,min_child_samples=45,
            subsample=.85,colsample_bytree=.80,reg_alpha=2.0,reg_lambda=15.0,
            random_state=seed,n_jobs=-1,verbosity=-1,force_col_wise=True,
        )
        lgb.fit(dev.loc[weak6_train,WEAK6_FEATURES],weak6_y.loc[weak6_train],sample_weight=weak6_weights,
                eval_set=[(dev.loc[weak6_tune,WEAK6_FEATURES],weak6_y.loc[weak6_tune])],
                eval_metric='average_precision',callbacks=[early_stopping(130,verbose=False),log_evaluation(0)])
        joblib.dump(lgb,lgb_path)
    weak6_lgb_models.append(lgb)
    weak6_history.append({'seed':seed,'cat_trees':int(cat.tree_count_),
                          'lgb_iteration':int(getattr(lgb,'best_iteration_',0) or lgb.n_estimators)})

dev['weak6_cat']=np.mean([model.predict_proba(dev[WEAK6_FEATURES])[:,1] for model in weak6_cat_models],axis=0)
dev['weak6_lgb']=np.mean([model.predict_proba(dev[WEAK6_FEATURES])[:,1] for model in weak6_lgb_models],axis=0)
dev['weak6_min']=np.minimum(dev.weak6_cat,dev.weak6_lgb)
dev['weak6_geom']=np.sqrt(np.clip(dev.weak6_cat,0,1)*np.clip(dev.weak6_lgb,0,1))
print(weak6_history)


In [ ]:
def weak_fault_mean_from_metric(metric):
    values=[metric['per_fault_episode_recall'].get(fault,np.nan) for fault in WEAK_TYPES]
    values=[value for value in values if not pd.isna(value)]
    return float(np.mean(values)) if values else 0.0

weak6_run_cache={}
visible6=dev.available_to_detector.eq(1)
for score_col in ['weak6_min','weak6_geom']:
  for threshold in [.10,.20,.30,.40,.50,.60,.70,.80]:
    for max_gap in [90,180,360]:
        full_run=pd.Series(0,index=dev.index,dtype=int)
        full_run.loc[visible6]=gap_aware_run_length(
            dev.loc[visible6],score_col,threshold,max_gap).astype(int)
        weak6_run_cache[(score_col,threshold,max_gap)]=full_run

def evaluate_weak6_candidate(score_col,threshold,min_points,max_gap):
    rescue=weak6_run_cache[(score_col,threshold,max_gap)].ge(min_points)
    rescue&=~dev.weather6_prediction
    candidate=dev.iteration5_reference|rescue
    rows=[]
    for block in WEATHER6_DISCOVERY_BLOCKS:
        mask=discovery&dev.dev_split.eq(block)
        part=dev.loc[mask].copy()
        part['baseline']=dev.loc[mask,'iteration5_reference'].to_numpy(bool)
        part['candidate']=candidate.loc[mask].to_numpy(bool)
        baseline=evaluate(part,'base_score','baseline'); metric=evaluate(part,'base_score','candidate')
        rows.append({
            'block':block,**metric,
            'point_f1_delta':metric['f1']-baseline['f1'],
            'event_f1_delta':metric['event_f1']-baseline['event_f1'],
            'weak_episode_recall':weak_fault_mean_from_metric(metric),
            'weak_episode_recall_delta':weak_fault_mean_from_metric(metric)-weak_fault_mean_from_metric(baseline),
        })
    return {
        'score_col':score_col,'threshold':threshold,'min_points':min_points,'max_gap_minutes':max_gap,
        'rows':rows,'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_point_f1_delta':min(row['point_f1_delta'] for row in rows),
        'mean_point_f1_delta':float(np.mean([row['point_f1_delta'] for row in rows])),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
        'mean_weak_recall_delta':float(np.mean([row['weak_episode_recall_delta'] for row in rows])),
    }

weak6_candidates=[]
for score_col in ['weak6_min','weak6_geom']:
  for threshold in [.10,.20,.30,.40,.50,.60,.70,.80]:
    for min_points in [2,3,4,6]:
      for max_gap in [90,180,360]:
        weak6_candidates.append(evaluate_weak6_candidate(score_col,threshold,min_points,max_gap))

def weak6_discovery_pass(row):
    return (row['min_precision']>=.75 and row['max_false_alarm']<=.02 and
            row['min_point_f1_delta']>=0 and row['min_event_f1_delta']>=-.01 and
            row['mean_point_f1_delta']>0 and row['mean_event_f1_delta']>=0 and
            row['mean_weak_recall_delta']>0)

weak6_frontier=pd.DataFrame([{key:value for key,value in row.items() if key!='rows'} for row in weak6_candidates])
weak6_frontier['passes_discovery']=weak6_frontier.apply(lambda row:weak6_discovery_pass(row.to_dict()),axis=1)
weak6_frontier.to_csv(ITER6_ROOT/'iteration6_weak_policy_frontier.csv',index=False)
weak6_feasible=[row for row in weak6_candidates if weak6_discovery_pass(row)]
proposed_weak6=(sorted(weak6_feasible,key=lambda row:(row['mean_weak_recall_delta'],
                                                       row['mean_point_f1_delta'],row['mean_event_f1_delta']),
                       reverse=True)[0] if weak6_feasible else None)

def weak6_confirmation_scope(mask,policy,scope):
    rescue=weak6_run_cache[(policy['score_col'],policy['threshold'],policy['max_gap_minutes'])].ge(policy['min_points'])
    rescue&=~dev.weather6_prediction
    part=dev.loc[mask].copy()
    part['baseline']=dev.loc[mask,'iteration5_reference'].to_numpy(bool)
    part['candidate']=(dev.loc[mask,'iteration5_reference']|rescue.loc[mask]).to_numpy(bool)
    baseline=evaluate(part,'base_score','baseline'); metric=evaluate(part,'base_score','candidate')
    return {
        'scope':scope,'precision':metric['precision'],
        'false_alarm_episodes_per_station_day':metric['false_alarm_episodes_per_station_day'],
        'point_f1':metric['f1'],'point_f1_delta':metric['f1']-baseline['f1'],
        'event_f1':metric['event_f1'],'event_f1_delta':metric['event_f1']-baseline['event_f1'],
        'weak_episode_recall':weak_fault_mean_from_metric(metric),
        'weak_episode_recall_delta':weak_fault_mean_from_metric(metric)-weak_fault_mean_from_metric(baseline),
    }

weak6_confirmation=[]
if proposed_weak6:
    weak6_confirmation.append(weak6_confirmation_scope(
        discovery&dev.dev_split.eq('block_oct_dec'),proposed_weak6,'oct_dec_discovery'))
    weak6_confirmation.append(weak6_confirmation_scope(
        pseudo&dev.dev_split.isin(POLICY_BLOCKS),proposed_weak6,'pseudo_unseen_all_2023'))
    weak6_confirmation_pass=all(
        row['precision']>=.75 and row['false_alarm_episodes_per_station_day']<=.02 and
        row['point_f1_delta']>=0 and row['event_f1_delta']>=-.01 and
        row['weak_episode_recall_delta']>=0 for row in weak6_confirmation)
else:
    weak6_confirmation_pass=False

if proposed_weak6 and weak6_confirmation_pass:
    WEAK6_STATUS='station_transfer_weak_rescue_confirmed'
    SELECTED_WEAK6={key:proposed_weak6[key] for key in ['score_col','threshold','min_points','max_gap_minutes']}
else:
    WEAK6_STATUS='no_station_transfer_gain_keep_iteration5_candidate'
    SELECTED_WEAK6={'score_col':'weak6_min','threshold':1.10,'min_points':999,'max_gap_minutes':180}

weak6_confirmation=pd.DataFrame(weak6_confirmation)
weak6_confirmation.to_csv(ITER6_ROOT/'iteration6_weak_confirmation.csv',index=False)
print('Weak transfer status:',WEAK6_STATUS,SELECTED_WEAK6)
display(weak6_frontier.sort_values(['passes_discovery','mean_weak_recall_delta','mean_point_f1_delta'],ascending=False).head(15))
display(weak6_confirmation)


## 26. Three-block ablation and result package

The final development candidate is materialized only when both discovery and confirmation gates pass. Failed experiments remain in the frontier files and do not silently alter the detector.


In [ ]:
if int(SELECTED_WEAK6['min_points'])<100:
    selected_weak6_rescue=weak6_run_cache[(SELECTED_WEAK6['score_col'],SELECTED_WEAK6['threshold'],
                                           SELECTED_WEAK6['max_gap_minutes'])].ge(SELECTED_WEAK6['min_points'])
    selected_weak6_rescue&=~dev.weather6_prediction
else:
    selected_weak6_rescue=pd.Series(False,index=dev.index)

dev['iteration6_candidate']=dev.iteration5_reference|selected_weak6_rescue
iteration6_rows=[]
for block in POLICY_BLOCKS:
    mask=dev.dev_split.eq(block)&dev.available_to_detector.eq(1)
    part=dev.loc[mask].copy()
    part['iteration5']=dev.loc[mask,'iteration5_reference'].to_numpy(bool)
    part['iteration6']=dev.loc[mask,'iteration6_candidate'].to_numpy(bool)
    for variant,pred_col in [('Iteration5 candidate','iteration5'),('Iteration6 candidate','iteration6')]:
        metric=evaluate(part,'base_score',pred_col)
        iteration6_rows.append({'block':block,'variant':variant,**metric,
                                'weak_episode_recall':weak_fault_mean_from_metric(metric)})

iteration6_ablation=pd.DataFrame(iteration6_rows)
iteration6_ablation.to_csv(ITER6_ROOT/'iteration6_multiblock_ablation.csv',index=False)
display(iteration6_ablation[['block','variant','precision','recall','f1','event_precision','event_recall',
                             'event_f1','weak_episode_recall','false_alarm_episodes_per_station_day']])

result6={
    'iteration':'06_communication_safety_weather_station_transfer',
    'device':DEVICE,'gpu':torch.cuda.get_device_name(0),
    'blind_2025_opened':False,'former_final_tests_opened':False,
    'development_years':[2022,2023],
    'pseudo_unseen_stations':PSEUDO_HOLDOUT_STATIONS,
    'communication':{
        'status':GAP_POLICY_STATUS,'selected_policy':SELECTED_GAP_POLICY,
        'safe_automatic_candidates':int(len(safe_gap_candidates)),
        'best_gap_policy':gap_frontier.iloc[0].to_dict(),
        'confirmation':gap_confirmation.to_dict('records'),
        'stream_action_used_by_detector':False,
    },
    'weather':{
        'status':WEATHER6_STATUS,'selected_policy':SELECTED_WEATHER6,
        'feature_count':len(WEATHER_INV_FEATURES),'model_history':weather6_history,
        'discovery_feasible_candidates':int(len(feasible_weather)),
        'confirmation':weather6_confirmation.to_dict('records'),
    },
    'weak_fault_transfer':{
        'status':WEAK6_STATUS,'selected_policy':SELECTED_WEAK6,
        'feature_count':len(WEAK6_FEATURES),'model_history':weak6_history,
        'discovery_feasible_candidates':int(len(weak6_feasible)),
        'confirmation':weak6_confirmation.to_dict('records'),
    },
    'selected_ablation':iteration6_ablation.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
    'promotion_rule':'No change unless discovery, October, pseudo-unseen, precision, false-alarm, point-F1 and event-F1 gates pass.',
}
(ITER6_ROOT/'iteration6_result_block.json').write_text(json.dumps(result6,indent=2,default=float))
(ITER6_ROOT/'iteration6_feature_contract.json').write_text(json.dumps({
    'weather_features':WEATHER_INV_FEATURES,'weak_features':WEAK6_FEATURES,
    'detector_observation_inputs':['temperature','pressure','relative_humidity'],
    'communication_metadata':['station_id','arrival_timestamp','optional_expected_cadence'],
    'forbidden':['dew_point','future_observation','blind_2025_labels'],
},indent=2))

print(json.dumps(result6,indent=2,default=float))
print('\nHISTORICAL ITERATION 6 FILES — continue running; do not return these yet:')
for filename in [
    'iteration6_result_block.json','iteration6_communication_frontier.csv',
    'iteration6_communication_confirmation.csv',
    'iteration6_weather_policy_frontier.csv','iteration6_weather_confirmation.csv',
    'iteration6_weak_policy_frontier.csv','iteration6_weak_confirmation.csv',
    'iteration6_multiblock_ablation.csv','iteration6_feature_contract.json',
]: print(ITER6_ROOT/filename)


## Historical Iteration 6 decision — continue to Iteration 7

Do not reopen the 2025 benchmark. If a challenger passes every development and pseudo-unseen gate, freeze it and create a new later benchmark before production promotion. If it fails, retain the prior detector and use the frontier evidence to decide whether the next development phase needs broader weather simulation or additional weak-fault episodes.


# Iteration 7 controlled development phase

The only model-observation inputs remain temperature, atmospheric pressure, and relative humidity. Station identifier, cluster, timestamps and neighbour metadata are routing/context metadata, not additional meteorological measurements. No future row, dew point, 2024 label or 2025 label is permitted.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import Dataset,DataLoader
import torch.nn.functional as F

ITER7_ROOT=DRIVE_ROOT/'experiments'/'iteration_07_climate_calibration_causal_tcn'
ITER7_ROOT.mkdir(parents=True,exist_ok=True)
assert UNLOCK_FINAL_TESTS is False,'Iteration 7 must keep all former final tests locked.'
assert 'blind_2025' not in str(ITER7_ROOT).lower()
assert result6['blind_2025_opened'] is False and result6['former_final_tests_opened'] is False
assert result6['weather']['status']=='no_confirmed_weather_gain_keep_iteration5_weather'
assert result6['weak_fault_transfer']['status']=='no_station_transfer_gain_keep_iteration5_candidate'

ITER7_DISCOVERY_BLOCKS=['block_may_jun','block_jul_sep']
ITER7_CONFIRMATION_SCOPES={
    'oct_dec_discovery':discovery&dev.dev_split.eq('block_oct_dec'),
    'pseudo_unseen_all_2023':pseudo&dev.dev_split.isin(POLICY_BLOCKS),
}
dev['iteration7_reference']=dev.iteration5_reference.astype(bool)
print('Iteration 7 root:',ITER7_ROOT)
print('Discovery stations:',len(DISCOVERY_STATIONS),'| pseudo-unseen:',PSEUDO_HOLDOUT_STATIONS)


## 27. Freeze the communication safety contract

An unknown-cadence silence is not identifiable as a station failure from archive timestamps alone. Iteration 7 carries the validated policy forward unchanged and emits a machine-readable adapter contract for the API/dashboard phase.


In [ ]:
ITER7_COMMUNICATION_CONTRACT={
    'version':'1.0',
    'automatic_duplicate_detection':True,
    'automatic_dropout_alert_requires':{
        'expected_cadence_seconds':'positive integer supplied by source adapter',
        'heartbeat_sla_seconds':'positive integer supplied by source adapter',
        'source_contract_verified':True,
    },
    'unknown_cadence_status':'unverified_data_gap_advisory',
    'unknown_cadence_is_sensor_fault':False,
    'maintenance_ticket_from_unknown_gap':False,
    'detector_observation_inputs':['temperature','pressure','relative_humidity'],
    'communication_metadata':['station_id','arrival_timestamp','expected_cadence_seconds','heartbeat_sla_seconds'],
}
assert SELECTED_GAP_POLICY['automatic_fault_alert_without_contract'] is False
(ITER7_ROOT/'iteration7_communication_contract.json').write_text(
    json.dumps(ITER7_COMMUNICATION_CONTRACT,indent=2))
display(pd.Series(ITER7_COMMUNICATION_CONTRACT,name='contract').to_frame())


## 28. Climate-cluster percentile calibration

The Iteration 6 weather models improved average F1 but shifted score distributions between stations. A fixed global threshold therefore helped some stations and hurt others. The calibrator below learns empirical score distributions from clean discovery-station history and maps each score to a climate-cluster percentile. Pseudo-unseen stations receive only their cluster reference; no station-specific label or threshold is fitted.


In [ ]:
WEATHER7_SCORE_COLUMNS=['weather6_cat','weather6_lgb','weather6_mean','weather6_geom']
WEATHER7_CONTEXT_CANDIDATES=[
    'regional_agreement_mean','regional_agreement_min','regional_agreeing_sensor_count',
    'regional_standardized_disagreement_max','regional_trend_disagreement_mean',
    'neighbor_temperature_agreement_fraction','neighbor_pressure_agreement_fraction',
    'neighbor_humidity_agreement_fraction','temperature_slope_3h','temperature_slope_6h',
    'pressure_slope_3h','pressure_slope_6h','humidity_slope_3h','humidity_slope_6h',
    'temperature_neighbor_residual_slope_3h','pressure_neighbor_residual_slope_3h',
    'humidity_neighbor_residual_slope_3h','temperature_robust_z_24h','pressure_robust_z_24h',
    'humidity_robust_z_24h',
]
WEATHER7_CONTEXT_FEATURES=[feature for feature in WEATHER7_CONTEXT_CANDIDATES if feature in dev.columns]
assert len(WEATHER7_CONTEXT_FEATURES)>=15

weather7_reference_mask=(
    discovery&dev.dev_split.eq('tune_model')&dev.is_weather_event.eq(0)&dev.is_anomaly.eq(0))

def fit_cluster_ecdf(frame,mask,score_columns):
    references={'cluster':{},'global':{}}
    for score_col in score_columns:
        global_values=np.sort(frame.loc[mask,score_col].dropna().to_numpy(float))
        assert len(global_values)>=100
        references['global'][score_col]=global_values
    for cluster,group in frame.loc[mask].groupby('cluster'):
        references['cluster'][str(cluster)]={}
        for score_col in score_columns:
            values=np.sort(group[score_col].dropna().to_numpy(float))
            references['cluster'][str(cluster)][score_col]=values
    return references

def apply_cluster_ecdf(frame,references,score_col):
    result=np.zeros(len(frame),dtype=float)
    for cluster,positions in frame.groupby('cluster',sort=False).indices.items():
        cluster_ref=references['cluster'].get(str(cluster),{}).get(score_col)
        reference=cluster_ref if cluster_ref is not None and len(cluster_ref)>=50 else references['global'][score_col]
        values=frame.iloc[positions][score_col].fillna(-np.inf).to_numpy(float)
        result[np.asarray(positions,dtype=int)]=np.searchsorted(reference,values,side='right')/max(len(reference),1)
    return np.clip(result,0,1)

weather7_ecdf=fit_cluster_ecdf(dev,weather7_reference_mask,WEATHER7_SCORE_COLUMNS)
for score_col in WEATHER7_SCORE_COLUMNS:
    dev[f'{score_col}_cluster_pct']=apply_cluster_ecdf(dev,weather7_ecdf,score_col)

WEATHER7_META_FEATURES=(
    [f'{score_col}_cluster_pct' for score_col in WEATHER7_SCORE_COLUMNS]+WEATHER7_CONTEXT_FEATURES)
assert not ({'temperature_value','pressure_value','humidity_value','station_id'}&set(WEATHER7_META_FEATURES))
print('Weather meta features:',len(WEATHER7_META_FEATURES))


In [ ]:
weather7_fit=discovery&dev.dev_split.eq('tune_model')
weather7_y=dev.is_weather_event.astype(int)
assert int(weather7_y.loc[weather7_fit].sum())>=20
weather7_weights=station_episode_balanced_weights(
    dev.loc[weather7_fit],weather7_y.loc[weather7_fit].to_numpy(bool))

def weather7_pipeline(penalty,C):
    return Pipeline([
        ('imputer',SimpleImputer(strategy='median',add_indicator=True)),
        ('scale',StandardScaler()),
        ('lr',LogisticRegression(
            penalty=penalty,C=C,solver='liblinear',class_weight=None,
            max_iter=2000,random_state=17)),
    ])

weather7_models={
    'weather7_l1':weather7_pipeline('l1',.30),
    'weather7_l2':weather7_pipeline('l2',.30),
}
for name,model in weather7_models.items():
    model.fit(dev.loc[weather7_fit,WEATHER7_META_FEATURES],weather7_y.loc[weather7_fit],
              lr__sample_weight=weather7_weights)
    dev[name]=model.predict_proba(dev[WEATHER7_META_FEATURES])[:,1]
    joblib.dump(model,ITER7_ROOT/f'{name}.joblib')

dev['weather7_blend']=(dev.weather7_l1+dev.weather7_l2)/2
weather7_nonzero={name:int(np.count_nonzero(model.named_steps['lr'].coef_))
                  for name,model in weather7_models.items()}
print('Weather calibrator non-zero coefficients:',weather7_nonzero)


In [ ]:
WEATHER7_SCORE_OPTIONS=['weather7_l1','weather7_l2','weather7_blend']

def weather7_candidate_summary(score_col,threshold):
    rows=[]
    for block in ITER7_DISCOVERY_BLOCKS:
        part=dev.loc[discovery&dev.dev_split.eq(block)].copy()
        part['baseline_pred']=part.cat_weather_mean.ge(float(WEATHER_GUARD_THRESHOLD))
        part['candidate_pred']=part[score_col].ge(float(threshold))
        baseline=weather_quality(part,'baseline_pred','cat_weather_mean')
        candidate=weather_quality(part,'candidate_pred',score_col)
        rows.append({
            'scope':block,**candidate,
            'f1_delta':candidate['weather_f1']-baseline['weather_f1'],
            'macro_delta':candidate['positive_station_macro_f1']-baseline['positive_station_macro_f1'],
        })
    return {
        'score_col':score_col,'threshold':float(threshold),'rows':rows,
        'min_f1_delta':float(min(row['f1_delta'] for row in rows)),
        'mean_f1_delta':float(np.mean([row['f1_delta'] for row in rows])),
        'min_macro_delta':float(min(row['macro_delta'] for row in rows)),
        'max_fault_to_weather':float(max(row['fault_to_weather_rate'] for row in rows)),
        'mean_weather_f1':float(np.mean([row['weather_f1'] for row in rows])),
    }

def weather7_discovery_pass(row):
    return (
        row['min_f1_delta']>=0 and row['mean_f1_delta']>0 and
        row['min_macro_delta']>=-.02 and row['max_fault_to_weather']<=.01)

weather7_candidates=[]
for score_col in WEATHER7_SCORE_OPTIONS:
    for threshold in np.linspace(.05,.95,37):
        weather7_candidates.append(weather7_candidate_summary(score_col,threshold))

weather7_frontier=pd.DataFrame([
    {key:value for key,value in row.items() if key!='rows'} for row in weather7_candidates])
weather7_frontier['passes_discovery']=weather7_frontier.apply(
    lambda row:weather7_discovery_pass(row.to_dict()),axis=1)
weather7_frontier.to_csv(ITER7_ROOT/'iteration7_weather_policy_frontier.csv',index=False)

weather7_feasible=[row for row in weather7_candidates if weather7_discovery_pass(row)]
proposed_weather7=(sorted(
    weather7_feasible,
    key=lambda row:(row['min_macro_delta'],row['mean_weather_f1'],row['mean_f1_delta']),
    reverse=True)[0] if weather7_feasible else None)
print('Weather discovery-feasible:',len(weather7_feasible),'of',len(weather7_candidates))
display(weather7_frontier.sort_values(
    ['passes_discovery','min_macro_delta','mean_weather_f1'],ascending=False).head(15))


In [ ]:
WEATHER7_CONFIRM_COLUMNS=[
    'scope','current_weather_precision','current_weather_recall','current_weather_f1',
    'current_weather_auprc','current_positive_station_macro_f1','current_fault_to_weather_rate',
    'candidate_weather_precision','candidate_weather_recall','candidate_weather_f1',
    'candidate_weather_auprc','candidate_positive_station_macro_f1','candidate_fault_to_weather_rate',
    'f1_delta','macro_delta']

weather7_confirmation=[]
if proposed_weather7:
    for scope,mask in ITER7_CONFIRMATION_SCOPES.items():
        weather7_confirmation.append(compare_weather_scope(
            mask,proposed_weather7['score_col'],proposed_weather7['threshold'],scope))

weather7_confirmation_frame=pd.DataFrame(weather7_confirmation,columns=WEATHER7_CONFIRM_COLUMNS)
weather7_confirmation_pass=(len(weather7_confirmation_frame)==2 and all(
    row['f1_delta']>=0 and row['macro_delta']>=-.02 and
    row['candidate_fault_to_weather_rate']<=.01
    for row in weather7_confirmation))

if proposed_weather7 and weather7_confirmation_pass:
    WEATHER7_STATUS='climate_calibrated_weather_confirmed'
    SELECTED_WEATHER7={
        'score_col':proposed_weather7['score_col'],
        'threshold':proposed_weather7['threshold']}
else:
    WEATHER7_STATUS='no_confirmed_weather_gain_keep_iteration5_weather'
    SELECTED_WEATHER7={'score_col':'cat_weather_mean','threshold':float(WEATHER_GUARD_THRESHOLD)}

weather7_confirmation_frame.to_csv(ITER7_ROOT/'iteration7_weather_confirmation.csv',index=False)
dev['weather7_prediction']=dev[SELECTED_WEATHER7['score_col']].ge(float(SELECTED_WEATHER7['threshold']))
print('Weather status:',WEATHER7_STATUS,SELECTED_WEATHER7)
display(weather7_confirmation_frame)


In [ ]:
weather7_station_rows=[]
for scope,mask in {
    **{block:discovery&dev.dev_split.eq(block) for block in ITER7_DISCOVERY_BLOCKS},
    **ITER7_CONFIRMATION_SCOPES,
}.items():
    for station,group in dev.loc[mask].groupby('station_id'):
        if group.is_weather_event.sum()==0:
            continue
        baseline=group.cat_weather_mean.ge(float(WEATHER_GUARD_THRESHOLD))
        candidate=group[SELECTED_WEATHER7['score_col']].ge(float(SELECTED_WEATHER7['threshold']))
        weather7_station_rows.append({
            'scope':scope,'station_id':str(station),'cluster':str(group.cluster.iloc[0]),
            'weather_rows':int(group.is_weather_event.sum()),
            'baseline_f1':float(f1_score(group.is_weather_event,baseline,zero_division=0)),
            'candidate_f1':float(f1_score(group.is_weather_event,candidate,zero_division=0)),
        })
weather7_station_metrics=pd.DataFrame(weather7_station_rows)
if len(weather7_station_metrics):
    weather7_station_metrics['f1_delta']=weather7_station_metrics.candidate_f1-weather7_station_metrics.baseline_f1
weather7_station_metrics.to_csv(ITER7_ROOT/'iteration7_weather_station_metrics.csv',index=False)
display(weather7_station_metrics.sort_values(['scope','f1_delta']).head(20))


## 29. Causal TCN for weak-fault sequence evidence

The tree models see a feature row at a time. Bias and drift are weak sequence phenomena, so Iteration 7 adds a compact causal TCN. Every window ends at the current observation; left padding is used at the start of a station history, and no future observation enters a window.

Raw temperature, raw pressure, raw humidity, calendar encodings and dew point are excluded. The TCN sees relative residuals, slopes, CUSUM, frozen runs, timing gaps and neighbour disagreement only.


In [ ]:
WEAK7_SEQUENCE_CANDIDATES=[
    'time_since_previous_minutes','gap_ratio','primary_missing_count','out_of_order_indicator',
    'temperature_delta1','temperature_rate_per_hour','temperature_rolling_mad_24h',
    'temperature_robust_z_24h','temperature_ewma_residual','temperature_frozen_run_length',
    'pressure_delta1','pressure_rate_per_hour','pressure_rolling_mad_24h',
    'pressure_robust_z_24h','pressure_ewma_residual','pressure_frozen_run_length',
    'humidity_delta1','humidity_rate_per_hour','humidity_rolling_mad_24h',
    'humidity_robust_z_24h','humidity_ewma_residual','humidity_frozen_run_length',
    'neighbor_temperature_residual','neighbor_pressure_residual','neighbor_humidity_residual',
    'temperature_slope_3h','temperature_slope_6h','temperature_slope_12h',
    'pressure_slope_3h','pressure_slope_6h','pressure_slope_12h',
    'humidity_slope_3h','humidity_slope_6h','humidity_slope_12h',
    'temperature_neighbor_residual_slope_3h','pressure_neighbor_residual_slope_3h',
    'humidity_neighbor_residual_slope_3h','temperature_cusum_positive','temperature_cusum_negative',
    'pressure_cusum_positive','pressure_cusum_negative','humidity_cusum_positive','humidity_cusum_negative',
    'temperature_monotonic_run','pressure_monotonic_run','humidity_monotonic_run',
    'regional_agreement_mean','regional_agreement_min','regional_standardized_disagreement_max',
    'regional_trend_disagreement_mean',
]
WEAK7_SEQUENCE_FEATURES=[feature for feature in WEAK7_SEQUENCE_CANDIDATES if feature in dev.columns]
assert len(WEAK7_SEQUENCE_FEATURES)>=40
assert not ({'temperature_value','pressure_value','humidity_value','temperature_lag1',
             'pressure_lag1','humidity_lag1','temperature_dewpoint_spread_c'}&set(WEAK7_SEQUENCE_FEATURES))

weak7_train_mask=discovery&dev.dev_split.eq('train')
weak7_tune_mask=discovery&dev.dev_split.eq('tune_model')
weak7_y=dev.anomaly_type.isin(WEAK_TYPES).astype(np.float32)

normalizer_source=dev.loc[weak7_train_mask,WEAK7_SEQUENCE_FEATURES].replace([np.inf,-np.inf],np.nan)
weak7_median=normalizer_source.median().fillna(0)
weak7_iqr=(normalizer_source.quantile(.75)-normalizer_source.quantile(.25)).replace(0,1).fillna(1)
weak7_matrix=((dev[WEAK7_SEQUENCE_FEATURES].replace([np.inf,-np.inf],np.nan)-weak7_median)/weak7_iqr)
weak7_matrix=weak7_matrix.clip(-20,20).fillna(0).to_numpy(np.float32)

WEAK7_WINDOW=24
weak7_station_arrays={}; weak7_row_station=np.empty(len(dev),dtype=object); weak7_row_position=np.zeros(len(dev),dtype=np.int32)
for station,group in dev.sort_values(['station_id','emitted_timestamp_utc']).groupby('station_id',sort=False):
    rows=group.index.to_numpy(int)
    weak7_station_arrays[str(station)]=weak7_matrix[rows]
    weak7_row_station[rows]=str(station)
    weak7_row_position[rows]=np.arange(len(rows),dtype=np.int32)

weak7_global_weights=np.ones(len(dev),dtype=np.float32)
train_rows_all=np.flatnonzero(weak7_train_mask.to_numpy(bool))
balanced_result=station_episode_balanced_weights(
    dev.loc[weak7_train_mask],weak7_y.loc[weak7_train_mask].to_numpy(bool))
# Iteration 6 returns the weight array directly. Keep tuple compatibility so this
# continuation also works if an older reconstruction helper returns (weights, audit).
balanced=balanced_result[0] if isinstance(balanced_result,tuple) else balanced_result
weak7_global_weights[train_rows_all]=balanced.astype(np.float32)
print('TCN sequence features:',len(WEAK7_SEQUENCE_FEATURES),'| window:',WEAK7_WINDOW)


In [ ]:
class Weak7WindowDataset(Dataset):
    def __init__(self,rows,include_weight=True):
        self.rows=np.asarray(rows,dtype=np.int64); self.include_weight=include_weight
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row=int(self.rows[index]); station=weak7_row_station[row]; position=int(weak7_row_position[row])
        source=weak7_station_arrays[station]
        start=max(0,position-WEAK7_WINDOW+1); window=source[start:position+1]
        if len(window)<WEAK7_WINDOW:
            window=np.pad(window,((WEAK7_WINDOW-len(window),0),(0,0)),mode='constant')
        x=torch.from_numpy(window.T.copy())
        y=torch.tensor(weak7_y.iloc[row],dtype=torch.float32)
        weight=torch.tensor(weak7_global_weights[row] if self.include_weight else 1.0,dtype=torch.float32)
        return x,y,weight

class CausalConvBlock(nn.Module):
    def __init__(self,channels,kernel,dilation,dropout):
        super().__init__()
        self.left=(kernel-1)*dilation
        self.conv=nn.Conv1d(channels,channels,kernel,dilation=dilation)
        self.norm=nn.GroupNorm(1,channels)
        self.drop=nn.Dropout(dropout)
    def forward(self,x):
        residual=x
        x=self.conv(F.pad(x,(self.left,0)))
        x=self.drop(F.gelu(self.norm(x)))
        return x+residual

class Weak7TCN(nn.Module):
    def __init__(self,input_channels,channels=48):
        super().__init__()
        self.input=nn.Conv1d(input_channels,channels,1)
        self.blocks=nn.Sequential(
            CausalConvBlock(channels,3,1,.10),
            CausalConvBlock(channels,3,2,.10),
            CausalConvBlock(channels,3,4,.10),
            CausalConvBlock(channels,3,8,.10),
        )
        self.head=nn.Sequential(nn.Linear(channels,24),nn.GELU(),nn.Dropout(.10),nn.Linear(24,1))
    def forward(self,x):
        z=self.blocks(self.input(x))
        return self.head(z[:,:,-1]).squeeze(1)

def weak7_training_rows(seed):
    rng=np.random.default_rng(seed)
    train_mask=weak7_train_mask.to_numpy(bool); positive=weak7_y.to_numpy(bool)
    positive_rows=np.flatnonzero(train_mask&positive)
    negative_rows=np.flatnonzero(train_mask&~positive)
    hard=(dev.is_anomaly.eq(1)|dev.is_weather_event.eq(1)|
          dev.weak6_geom.ge(dev.loc[weak7_train_mask,'weak6_geom'].quantile(.90))).to_numpy(bool)
    hard_rows=np.flatnonzero(train_mask&~positive&hard)
    random_pool=np.setdiff1d(negative_rows,hard_rows,assume_unique=False)
    random_count=min(len(random_pool),max(4*len(positive_rows),20000))
    random_rows=rng.choice(random_pool,size=random_count,replace=False)
    return np.unique(np.concatenate([positive_rows,hard_rows,random_rows]))

def predict_weak7_tcn(model,rows,batch_size=2048):
    loader=DataLoader(Weak7WindowDataset(rows,include_weight=False),batch_size=batch_size,
                      shuffle=False,num_workers=0,pin_memory=True)
    output=[]; model.eval()
    with torch.no_grad():
        for x,_,_ in loader:
            output.append(torch.sigmoid(model(x.to(DEVICE,non_blocking=True))).cpu().numpy())
    return np.concatenate(output)


In [ ]:
def train_weak7_tcn(seed,max_epochs=20,patience=4):
    torch.manual_seed(seed); np.random.seed(seed)
    train_rows=weak7_training_rows(seed)
    tune_rows=np.flatnonzero(weak7_tune_mask.to_numpy(bool))
    train_loader=DataLoader(Weak7WindowDataset(train_rows),batch_size=512,shuffle=True,
                            num_workers=0,pin_memory=True)
    model=Weak7TCN(len(WEAK7_SEQUENCE_FEATURES)).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=2e-4)
    best_state=None; best_auprc=-1; wait=0; history=[]
    for epoch in range(1,max_epochs+1):
        model.train(); losses=[]
        for x,y,weight in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE); weight=weight.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x)
            raw=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
            probability=torch.sigmoid(logits)
            pt=torch.where(y.gt(.5),probability,1-probability)
            loss=((1-pt).pow(1.5)*raw*weight).mean()
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0); optimizer.step()
            losses.append(float(loss.detach().cpu()))
        tune_score=predict_weak7_tcn(model,tune_rows)
        tune_auprc=float(average_precision_score(weak7_y.iloc[tune_rows],tune_score))
        history.append({'seed':seed,'epoch':epoch,'loss':float(np.mean(losses)),'tune_auprc':tune_auprc})
        print(f'seed={seed} epoch={epoch} loss={np.mean(losses):.5f} tune_auprc={tune_auprc:.5f}')
        if tune_auprc>best_auprc+1e-4:
            best_auprc=tune_auprc; wait=0
            best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state); model.to(DEVICE).eval()
    torch.save({'state_dict':best_state,'features':WEAK7_SEQUENCE_FEATURES,'window':WEAK7_WINDOW,
                'median':weak7_median.to_dict(),'iqr':weak7_iqr.to_dict(),'seed':seed},
               ITER7_ROOT/f'weak7_causal_tcn_seed{seed}.pt')
    return model,history,best_auprc

WEAK7_TCN_SEEDS=[17,41]
weak7_tcn_models=[]; weak7_training_history=[]; weak7_best_auprc=[]
for seed in WEAK7_TCN_SEEDS:
    model,history,best=train_weak7_tcn(seed)
    weak7_tcn_models.append(model); weak7_training_history.extend(history); weak7_best_auprc.append(best)

all_rows=np.arange(len(dev),dtype=np.int64)
dev['weak7_tcn_raw']=np.mean([predict_weak7_tcn(model,all_rows) for model in weak7_tcn_models],axis=0)
pd.DataFrame(weak7_training_history).to_csv(ITER7_ROOT/'iteration7_tcn_training_history.csv',index=False)
print('TCN best tune AUPRC:',weak7_best_auprc)


In [ ]:
weak7_platt=LogisticRegression(C=.5,solver='lbfgs',class_weight='balanced',max_iter=1000,random_state=17)
weak7_platt.fit(dev.loc[weak7_tune_mask,['weak7_tcn_raw']],weak7_y.loc[weak7_tune_mask])
dev['weak7_tcn']=weak7_platt.predict_proba(dev[['weak7_tcn_raw']])[:,1]
joblib.dump(weak7_platt,ITER7_ROOT/'weak7_tcn_platt.joblib')

dev['weak7_tree']=np.sqrt(np.clip(dev.weak6_cat,0,1)*np.clip(dev.weak6_lgb,0,1))
dev['weak7_min']=np.minimum(dev.weak7_tree,dev.weak7_tcn)
dev['weak7_geom']=np.sqrt(np.clip(dev.weak7_tree,0,1)*np.clip(dev.weak7_tcn,0,1))
dev['weak7_blend']=.65*dev.weak7_tree+.35*dev.weak7_tcn
print(dev[['weak7_tree','weak7_tcn','weak7_min','weak7_geom','weak7_blend']].describe().T)


## 30. Incident-level consensus policy

A weak-fault rescue is allowed only when its score persists across multiple causal observations and the selected weather gate does not identify a coherent meteorological event. Thresholds are discovered on May–September only. October and pseudo-unseen stations are never searched.


In [ ]:
WEAK7_SCORE_OPTIONS=['weak7_min','weak7_geom','weak7_blend']
WEAK7_THRESHOLDS=[.10,.15,.20,.25,.30,.35,.40,.50,.60,.70,.80]
WEAK7_MIN_POINTS=[2,3,4,6]
WEAK7_MAX_GAPS=[90,180]
weak7_run_cache={}
for score_col in WEAK7_SCORE_OPTIONS:
    for threshold in WEAK7_THRESHOLDS:
        for max_gap in WEAK7_MAX_GAPS:
            weak7_run_cache[(score_col,threshold,max_gap)]=gap_aware_run_length(
                dev,score_col,threshold,max_gap)

def evaluate_weak7_candidate(score_col,threshold,min_points,max_gap):
    rescue=weak7_run_cache[(score_col,threshold,max_gap)].ge(min_points)&~dev.weather7_prediction
    candidate=dev.iteration7_reference|rescue
    rows=[]
    for block in ITER7_DISCOVERY_BLOCKS:
        mask=discovery&dev.dev_split.eq(block)
        part=dev.loc[mask].copy()
        part['baseline']=dev.loc[mask,'iteration7_reference'].to_numpy(bool)
        part['candidate']=candidate.loc[mask].to_numpy(bool)
        baseline=evaluate(part,'base_score','baseline'); metric=evaluate(part,'base_score','candidate')
        rows.append({
            'scope':block,**metric,
            'point_f1_delta':metric['f1']-baseline['f1'],
            'event_f1_delta':metric['event_f1']-baseline['event_f1'],
            'weak_episode_recall':weak_fault_mean_from_metric(metric),
            'weak_episode_recall_delta':weak_fault_mean_from_metric(metric)-weak_fault_mean_from_metric(baseline),
        })
    return {
        'score_col':score_col,'threshold':threshold,'min_points':min_points,'max_gap_minutes':max_gap,
        'rows':rows,'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_point_f1_delta':min(row['point_f1_delta'] for row in rows),
        'mean_point_f1_delta':float(np.mean([row['point_f1_delta'] for row in rows])),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
        'min_weak_recall_delta':min(row['weak_episode_recall_delta'] for row in rows),
        'mean_weak_recall_delta':float(np.mean([row['weak_episode_recall_delta'] for row in rows])),
    }

def weak7_discovery_pass(row):
    return (
        row['min_precision']>=.75 and row['max_false_alarm']<=.02 and
        row['min_point_f1_delta']>=0 and row['min_event_f1_delta']>=-.01 and
        row['mean_point_f1_delta']>0 and row['mean_event_f1_delta']>=0 and
        row['min_weak_recall_delta']>=0 and row['mean_weak_recall_delta']>0)

weak7_candidates=[]
for score_col in WEAK7_SCORE_OPTIONS:
  for threshold in WEAK7_THRESHOLDS:
    for min_points in WEAK7_MIN_POINTS:
      for max_gap in WEAK7_MAX_GAPS:
        weak7_candidates.append(evaluate_weak7_candidate(score_col,threshold,min_points,max_gap))

weak7_frontier=pd.DataFrame([
    {key:value for key,value in row.items() if key!='rows'} for row in weak7_candidates])
weak7_frontier['passes_discovery']=weak7_frontier.apply(
    lambda row:weak7_discovery_pass(row.to_dict()),axis=1)
weak7_frontier.to_csv(ITER7_ROOT/'iteration7_weak_policy_frontier.csv',index=False)
weak7_feasible=[row for row in weak7_candidates if weak7_discovery_pass(row)]
proposed_weak7=(sorted(
    weak7_feasible,
    key=lambda row:(row['mean_weak_recall_delta'],row['mean_point_f1_delta'],row['mean_event_f1_delta']),
    reverse=True)[0] if weak7_feasible else None)
print('Weak discovery-feasible:',len(weak7_feasible),'of',len(weak7_candidates))
display(weak7_frontier.sort_values(
    ['passes_discovery','mean_weak_recall_delta','mean_point_f1_delta'],ascending=False).head(15))


In [ ]:
def weak7_confirmation_scope(mask,policy,scope):
    rescue=weak7_run_cache[(policy['score_col'],policy['threshold'],policy['max_gap_minutes'])].ge(policy['min_points'])
    rescue&=~dev.weather7_prediction
    part=dev.loc[mask].copy()
    part['baseline']=dev.loc[mask,'iteration7_reference'].to_numpy(bool)
    part['candidate']=(dev.loc[mask,'iteration7_reference']|rescue.loc[mask]).to_numpy(bool)
    baseline=evaluate(part,'base_score','baseline'); metric=evaluate(part,'base_score','candidate')
    return {
        'scope':scope,'precision':metric['precision'],
        'false_alarm_episodes_per_station_day':metric['false_alarm_episodes_per_station_day'],
        'point_f1':metric['f1'],'point_f1_delta':metric['f1']-baseline['f1'],
        'event_f1':metric['event_f1'],'event_f1_delta':metric['event_f1']-baseline['event_f1'],
        'weak_episode_recall':weak_fault_mean_from_metric(metric),
        'weak_episode_recall_delta':weak_fault_mean_from_metric(metric)-weak_fault_mean_from_metric(baseline),
    }

WEAK7_CONFIRM_COLUMNS=[
    'scope','precision','false_alarm_episodes_per_station_day','point_f1','point_f1_delta',
    'event_f1','event_f1_delta','weak_episode_recall','weak_episode_recall_delta']
weak7_confirmation=[]
if proposed_weak7:
    for scope,mask in ITER7_CONFIRMATION_SCOPES.items():
        weak7_confirmation.append(weak7_confirmation_scope(mask,proposed_weak7,scope))

weak7_confirmation_frame=pd.DataFrame(weak7_confirmation,columns=WEAK7_CONFIRM_COLUMNS)
weak7_confirmation_pass=(len(weak7_confirmation_frame)==2 and all(
    row['precision']>=.75 and row['false_alarm_episodes_per_station_day']<=.02 and
    row['point_f1_delta']>=0 and row['event_f1_delta']>=-.01 and
    row['weak_episode_recall_delta']>=0 for row in weak7_confirmation) and
    next(row for row in weak7_confirmation if row['scope']=='pseudo_unseen_all_2023')['weak_episode_recall_delta']>0)

if proposed_weak7 and weak7_confirmation_pass:
    WEAK7_STATUS='causal_tcn_consensus_confirmed'
    SELECTED_WEAK7={key:proposed_weak7[key] for key in ['score_col','threshold','min_points','max_gap_minutes']}
else:
    WEAK7_STATUS='no_confirmed_tcn_transfer_keep_iteration5_candidate'
    SELECTED_WEAK7={'score_col':'weak7_min','threshold':1.10,'min_points':999,'max_gap_minutes':180}

weak7_confirmation_frame.to_csv(ITER7_ROOT/'iteration7_weak_confirmation.csv',index=False)
print('Weak status:',WEAK7_STATUS,SELECTED_WEAK7)
display(weak7_confirmation_frame)


## 31. Final development ablation, fault coverage and integrity receipt

The selected candidate is materialized only after every confirmation gate passes. Otherwise the no-op policy preserves the Iteration 5 development reference exactly. A new external blind benchmark may be opened only after this notebook has been reviewed and the candidate frozen.


In [ ]:
if int(SELECTED_WEAK7['min_points'])<100:
    selected_weak7_rescue=weak7_run_cache[(
        SELECTED_WEAK7['score_col'],SELECTED_WEAK7['threshold'],SELECTED_WEAK7['max_gap_minutes'])].ge(
            SELECTED_WEAK7['min_points'])&~dev.weather7_prediction
else:
    selected_weak7_rescue=pd.Series(False,index=dev.index)

dev['iteration7_candidate']=dev.iteration7_reference|selected_weak7_rescue
iteration7_rows=[]; iteration7_combined=[]
for block in POLICY_BLOCKS:
    mask=dev.dev_split.eq(block)&dev.available_to_detector.eq(1)
    part=dev.loc[mask].copy()
    part['reference']=dev.loc[mask,'iteration7_reference'].to_numpy(bool)
    part['candidate']=dev.loc[mask,'iteration7_candidate'].to_numpy(bool)
    for variant,pred_col in [('Iteration5 reference','reference'),('Iteration7 candidate','candidate')]:
        metric=evaluate(part,'base_score',pred_col)
        iteration7_rows.append({
            'block':block,'variant':variant,**metric,
            'weak_episode_recall':weak_fault_mean_from_metric(metric)})
    iteration7_combined.append(part)

iteration7_ablation=pd.DataFrame(iteration7_rows)
iteration7_ablation.to_csv(ITER7_ROOT/'iteration7_multiblock_ablation.csv',index=False)
display(iteration7_ablation[['block','variant','precision','recall','f1','event_precision','event_recall',
                             'event_f1','weak_episode_recall','false_alarm_episodes_per_station_day']])

iteration7_all=pd.concat(iteration7_combined,ignore_index=True)
iteration7_all['pred']=iteration7_all.candidate
iteration7_fault_recall=(pd.Series(
    event_metrics(iteration7_all,'pred')['per_fault_episode_recall'],name='episode_recall')
    .sort_values().rename_axis('anomaly_type').reset_index())
iteration7_fault_recall.to_csv(ITER7_ROOT/'iteration7_fault_episode_recall.csv',index=False)
display(iteration7_fault_recall)


In [ ]:
iteration7_feature_contract={
    'detector_observation_inputs':['temperature','pressure','relative_humidity'],
    'routing_metadata':['station_id','timestamp','cluster','latitude','longitude'],
    'communication_metadata':['arrival_timestamp','optional_expected_cadence','optional_heartbeat_sla'],
    'weather_meta_features':WEATHER7_META_FEATURES,
    'weak_sequence_features':WEAK7_SEQUENCE_FEATURES,
    'sequence_window_observations':WEAK7_WINDOW,
    'causal_window':'current and previous observations only; left padded; no future rows',
    'forbidden':['dew_point','future_observation','2024_labels','blind_2025_labels'],
}
(ITER7_ROOT/'iteration7_feature_contract.json').write_text(
    json.dumps(iteration7_feature_contract,indent=2))

result7={
    'iteration':'07_climate_calibration_causal_tcn',
    'device':DEVICE,'gpu':torch.cuda.get_device_name(0),
    'blind_2025_opened':False,'former_final_tests_opened':False,
    'development_years':[2022,2023],
    'pseudo_unseen_stations':PSEUDO_HOLDOUT_STATIONS,
    'communication':{
        'status':'heartbeat_contract_frozen','contract':ITER7_COMMUNICATION_CONTRACT},
    'weather':{
        'status':WEATHER7_STATUS,'selected_policy':SELECTED_WEATHER7,
        'meta_feature_count':len(WEATHER7_META_FEATURES),
        'regularization_nonzero_coefficients':weather7_nonzero,
        'discovery_feasible_candidates':int(len(weather7_feasible)),
        'confirmation':weather7_confirmation_frame.to_dict('records')},
    'weak_fault_transfer':{
        'status':WEAK7_STATUS,'selected_policy':SELECTED_WEAK7,
        'sequence_feature_count':len(WEAK7_SEQUENCE_FEATURES),'window':WEAK7_WINDOW,
        'tcn_seeds':WEAK7_TCN_SEEDS,'best_tune_auprc':weak7_best_auprc,
        'discovery_feasible_candidates':int(len(weak7_feasible)),
        'confirmation':weak7_confirmation_frame.to_dict('records')},
    'selected_ablation':iteration7_ablation.drop(
        columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
    'promotion_rule':(
        'No change unless discovery, October, pseudo-unseen, precision, false-alarm, '
        'point-F1, event-F1, station-macro and pseudo-unseen weak-recall gates pass.'),
}
(ITER7_ROOT/'iteration7_result_block.json').write_text(
    json.dumps(result7,indent=2,default=float))

integrity7={
    'blind_2025_opened':False,'former_final_tests_opened':False,
    'development_partitions':['2022 train','2023 tune/discovery/confirmation'],
    'pseudo_unseen_stations':PSEUDO_HOLDOUT_STATIONS,
    'future_features_used':False,'dew_point_used':False,
    'communication_stream_action_used_by_detector':False,
    'result_sha256':hashlib.sha256((ITER7_ROOT/'iteration7_result_block.json').read_bytes()).hexdigest(),
    'feature_contract_sha256':hashlib.sha256((ITER7_ROOT/'iteration7_feature_contract.json').read_bytes()).hexdigest(),
}
(ITER7_ROOT/'iteration7_integrity_receipt.json').write_text(
    json.dumps(integrity7,indent=2))

print(json.dumps(result7,indent=2,default=float))
print('\nSEND BACK THESE ITERATION 7 FILES:')
for filename in [
    'iteration7_result_block.json','iteration7_integrity_receipt.json',
    'iteration7_communication_contract.json','iteration7_weather_policy_frontier.csv',
    'iteration7_weather_confirmation.csv','iteration7_weather_station_metrics.csv',
    'iteration7_tcn_training_history.csv','iteration7_weak_policy_frontier.csv',
    'iteration7_weak_confirmation.csv','iteration7_multiblock_ablation.csv',
    'iteration7_fault_episode_recall.csv','iteration7_feature_contract.json',
]: print(ITER7_ROOT/filename)


## Iteration 7 stop rule

Return the twelve files printed above. Do not open or tune against the 2025 benchmark. We will first audit discovery-versus-confirmation transfer, per-station weather stability, weak-fault event recall, false alarms, TCN convergence and the integrity receipt. Only a fully confirmed candidate will be frozen for a genuinely new blind evaluation.
